## 옵션 가격결정을 위한 유한차분법 (Finite Differences Option Pricing for Quant Finance)

##### ▶️ 관련 Quant Guild 영상 (영어)

*먼저 알아두면 좋은 확률미적분:*
-  [Itô's Lemma Clearly and Visually Explained](https://youtu.be/TgBzqdN24fo): 이토 보조정리
-  [Itô Integration Clearly and Visually Explained](https://youtu.be/dUvZ8m3QpeI): 이토 적분
- [Stochastic Differential Equations for Quant Finance](https://youtu.be/qDAeSC40ZJE): 확률미분방정식

*왜 기댓값만으로 가격을 매길 수 있는지, 그리고 실무에서의 문제점:*
- [Expected Stock Returns Don't Exist](https://youtu.be/iXNSBn5xqrA)
- [What Does AI Actually Learn](https://youtu.be/tX7b2KT63WQ)
- [How to Trade with an Edge](https://youtu.be/NlqpDB2BhxE)

*마켓메이킹과 트레이딩 응용:*
- [How to Trade with the Black-Scholes Model](https://youtu.be/0x-Pc-Z3wu4)
- [Trading with the Black-Scholes Implied Volatility Surface](https://youtu.be/YH0tWpBaKGs)
- [How to Price Exotic Options](https://youtu.be/hsot26myYYM)

###### ______________________________________________________________________________________________________________________________________

##### [📚 Quant Guild Library: 다른 주피터 노트북 모음](https://github.com/romanmichaelpaolucci/Quant-Guild-Library)

##### [🚀 Quant Guild](https://quantguild.com)

##### [📅 Quant Guild 라이브 강의](https://quantguild.com/live-classes)

---

**표지 그림:** 곡선 $f(x) = x^2$ (초록)과 여러 점에서 그은 접선(파랑)입니다. 접선의 기울기가 그 점에서의 미분값 $f'(x)$ 입니다.
이 노트북 전체의 아이디어가 이 그림에 담겨 있습니다. **각 점의 기울기(미분)만 알아도, 시작점에서 기울기를 따라 한 걸음씩 가면 곡선 전체를 복원할 수 있다.** 옵션 가격도 이렇게 구합니다.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Generate x values for the parabola
x = np.linspace(-2, 5, 200)

# Create the parabola function
def f(x):
    return x**2

# Calculate derivative approximation
h = 0.0001
def approx_derivative(x0):
    return (f(x0 + h) - f(x0))/h

# Create figure
fig = go.Figure()

# Plot the actual parabola
fig.add_trace(
    go.Scatter(
        x=x,
        y=f(x),
        mode='lines',
        line=dict(color='rgb(57, 255, 20)', width=3),
        name='Original Function f(x) = x²'
    )
)

# Add tangent lines at many points from x=0 to x=5
num_points = 20  # Increase number of points for smoother transition
points = np.linspace(0, 5, num_points)
base_opacity = 1
opacity_step = 0.9 / len(points)

# Add first tangent line that will appear in legend
x0 = points[0]
slope = approx_derivative(x0)
tangent = slope * (x - x0) + f(x0)
fig.add_trace(
    go.Scatter(
        x=x,
        y=tangent,
        mode='lines',
        line=dict(color='rgb(0, 191, 255)', width=2),
        name='Approximating Tangent Lines'
    )
)

# Add remaining tangent lines with decreasing opacity
for i, x0 in enumerate(points[1:]):
    slope = approx_derivative(x0)
    tangent = slope * (x - x0) + f(x0)
    opacity = base_opacity - (i * opacity_step)
    fig.add_trace(
        go.Scatter(
            x=x,
            y=tangent,
            mode='lines',
            line=dict(
                color=f'rgba(0, 191, 255, {opacity})',
                width=2
            ),
            showlegend=False
        )
    )

# Update layout
fig.update_layout(
    width=800,
    height=500,
    title_text='f(x) = x² and its Tangent Lines',
    title_x=0.5,
    title_font_size=20,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

# Update axes with fixed ranges to show full range
fig.update_xaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)',
    title_text='x',
    range=[-2, 5]  # Extended range to show full parabola
)
fig.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)',
    title_text='f(x)',
    range=[-2, 25]  # Extended range to show full parabola height
)

fig.show()


---

### 📖 목차

#### 1.) 🎯 퀀트 금융에서 미분방정식이 왜 나오나

- 가격결정 미분방정식이란

- 해석해(공식) vs 수치해(근사)

#### 2.) 🌊 상미분방정식(ODE) 근사하기

- 미분의 정의

- 정의를 그대로 써서 미분 근사하기

- 오일러 방법과 유한차분

- ODE를 수치적으로 풀기: 코드와 시각화

#### 3.) 🔥 편미분방정식(PDE) 근사하기: 열방정식

- 해를 눈으로 보기

- 수치적으로 풀기: 코드와 시각화

- 안정성 조건은 왜 필요한가

#### 4.) 📝 Black-Scholes PDE 근사하기

- explicit 방법: 만기 payoff에서 시작해 시간을 거꾸로

- 경계조건 (원본 버전의 버그)

- 🧮 격자 계산을 칸 단위로 시각화

- 왜 되는가: 이 계산은 삼항트리다

- 안정성, 수렴, PDE vs 몬테카를로 vs 공식

- 💡 그래서 왜 유한차분을 쓰나: 그릭스와 아메리칸 옵션

#### 5.) 🇰🇷 실전 예제: 스텝다운 ELS (기초자산 1개)

- 상품 구조와 대표 시나리오

- 격자에 올리는 방법: 뒤로 계산, 평가일 덮어쓰기, 낙인은 격자 두 장

- 몬테카를로 검증과 시나리오별 확률

- 격자 안 들여다보기 (히트맵, 애니메이션)

- 오늘의 가격곡선, 델타, 공정 쿠폰

#### 6.) 💭 마무리와 다음 주제

### 🗺️ 시작하기 전에: 이 노트북은 왜 필요한가

**출발 질문: 옵션 가격은 어떻게 계산할까?**

앞의 Black-Scholes 노트북에서는 공식 한 줄 $C = SN(d_1) - Ke^{-rT}N(d_2)$ 로 가격을 구했습니다. 그런데 이런 공식은 **아주 운이 좋은 경우**에만 존재합니다.

| 상품 / 모델 | 닫힌 공식이 있나? |
|---|---|
| BS 모델의 유럽형 콜·풋 | ✅ 있음 |
| 아메리칸 옵션 (만기 전 아무 때나 행사 가능) | ❌ 없음 |
| 이산 관찰 배리어 옵션, 이산 배당이 있는 옵션 등 | ❌ 대부분 없음 |
| Local volatility 모델 (실무에서 변동성 스마일을 맞추는 모델) | ❌ 없음 |
| Heston 같은 확률변동성 모델의 이색옵션 | ❌ 없음 |

실무에서 거래되는 상품 대부분은 공식이 없습니다. 그럼 가격을 어떻게 매길까요?

**핵심 사실:** 모델을 정하고 헤지 논리를 적용하면, 가격함수 $V(S,t)$ 는 항상 어떤 **미분방정식(PDE)**을 만족합니다. 공식은 없어도 **방정식은 있습니다.** 그러니 방정식을 컴퓨터로 근사해서 풀면 됩니다. 이 노트북에서 배우는 방법이 **유한차분법**(finite differences)입니다.

**유한차분법이 실무에서 쓰이는 이유**
1. **공식이 없는 상품도 풀 수 있다.** 예를 들어 아메리칸 옵션은 코드 한 줄만 추가하면 됩니다(4장 마지막 셀).
2. **한 번 풀면 모든 주가·모든 시점의 가격이 한꺼번에 나온다.** 격자 전체가 답이라서 델타·감마·세타 같은 **그릭스**(헤지에 필요한 민감도)도 격자에서 바로 읽힙니다. 몬테카를로는 주가 한 점의 가격만 주고, 그릭스를 구하려면 시뮬레이션을 더 돌려야 합니다.
3. **기초자산이 1~2개인 문제에서는 빠르고 정확하다.** 몬테카를로보다 훨씬 적은 계산으로 소수점 여러 자리까지 맞습니다.

(반대로 기초자산이 많은 고차원 문제에서는 격자 크기가 폭발해서 몬테카를로를 씁니다. 두 방법은 경쟁이 아니라 역할 분담입니다.)

**그런데 왜 공식이 있는 BS로 연습하나?** **정답을 알기 때문입니다.** 새로운 계산 방법은 정답을 아는 문제에서 먼저 검증해야 합니다. BS에서 공식과 맞는 걸 확인해야 공식이 없는 문제에서도 결과를 믿을 수 있습니다.

**이 노트북의 순서와 그 이유**
1. **ODE** (도함수 $f'(x)$ 만 주어진 1차원 문제): 가장 단순한 형태에서 "미분을 차분으로 바꾸고 한 칸씩 전진"이라는 아이디어를 익힙니다.
2. **열방정식** (변수가 $x, t$ 두 개인 PDE): 같은 아이디어를 격자(표)로 확장하고, **안정성**이라는 새 문제를 만납니다. BS PDE는 변수를 바꾸면 열방정식이 되므로 딱 맞는 연습문제입니다.
3. **BS PDE**: 진짜 목표입니다. 만기 payoff에서 시작해 시간을 거꾸로 격자를 채워 오늘의 가격을 구합니다.

---

In [ ]:
%%html
<style>
/* Overwrite the hard-coded white background for ipywidgets */
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
/* Set widget foreground text and color to match the VS Code dark theme */
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>

##### 1.) 🎯 퀀트 금융에서 미분방정식이 왜 나오나

**🎯 이 섹션의 목적:** 옵션 가격이 왜 "미분방정식의 해"가 되는지 이해하고, 그 해를 공식으로 구할 수 있을 때와 없을 때가 어떻게 다른지 봅니다.

**가격이 미분방정식이 되는 이유 (직관)**

옵션을 하나 팔고, 기초자산 주식을 적당량(델타만큼) 사서 **계속** 헤지한다고 해 봅시다. 주가가 조금 오르면 옵션에서 잃는 만큼 주식에서 벌고, 내리면 그 반대입니다. 헤지를 연속적으로 하면 주가의 무작위 움직임에서 오는 손익이 **정확히 상쇄**되어 포트폴리오가 무위험이 됩니다.

무위험 포트폴리오는 무위험이자율 $r$ 만큼만 벌어야 합니다. 더 벌면 공짜로 돈을 버는 차익거래가 생기기 때문입니다. 이 조건을 식으로 쓰면 가격함수 $V(S,t)$ 의 미분들 사이의 관계식, 즉 **미분방정식**이 나옵니다.

**무작위성은 어디로 갔나?** 헤지로 상쇄되어 사라졌습니다. 그래서 PDE 안에는 확률변수가 없고, 얼마나 흔들리는지를 나타내는 $\sigma$ 만 남습니다.

**Black-Scholes PDE:**

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0$$

각 항의 뜻:
- $\frac{\partial V}{\partial t}$ (세타): 시간이 흐르면서 옵션 가치가 변하는 속도
- $\frac12\sigma^2S^2\frac{\partial^2 V}{\partial S^2}$ (감마 항): 주가가 흔들린다는 사실 자체에서 오는 가치 (옵션 가격곡선의 볼록성)
- $rS\frac{\partial V}{\partial S}$ (델타 항): 위험중립 세계에서 주가가 이자율 $r$ 로 자라는 효과
- $-rV$: 할인 (미래 돈의 현재가치)

**Heston PDE** (변동성 자체도 무작위로 움직이는 모델):

$$\frac{\partial V}{\partial t} + \frac{1}{2}\nu S^2\frac{\partial^2 V}{\partial S^2} + \rho\sigma\nu S\frac{\partial^2 V}{\partial S\partial \nu} + \frac{1}{2}\sigma^2\nu\frac{\partial^2 V}{\partial \nu^2} + rS\frac{\partial V}{\partial S} + \kappa(\theta-\nu)\frac{\partial V}{\partial \nu} - rV = 0$$

> 📌 **수정:** 원본은 $\frac12\sigma^2S^2 V_{SS}$, $\frac12\nu V_{\nu\nu}$ 로 σ와 ν가 뒤바뀌어 있었습니다. Heston에서 $\nu$ 는 주가의 **분산**(시간에 따라 움직이는 상태변수)이고, $\sigma$ 는 **분산의 변동성**(vol of vol)입니다. 그래서 주가 확산항은 $\frac12\nu S^2$, 분산 확산항은 $\frac12\sigma^2\nu$ 입니다. BS PDE와 비교하면 상수 $\sigma^2$ 자리에 움직이는 $\nu$ 가 들어가고, $\nu$ 방향의 미분항들이 추가된 형태입니다. 변수가 하나 늘어서($S$, $\nu$, $t$) 풀기가 훨씬 어려워집니다.

이 방정식을 풀면 그 모델에서의 **가격함수**가 나옵니다. 이런 논리는 일반 옵션(vanilla)과 이색옵션(exotic) 모두에 적용됩니다. 이 노트북에서는 **경로에 의존하지 않는 유럽형 옵션**(만기 주가만으로 payoff가 정해짐)에 집중합니다.

##### 🔍 예제: 가격결정 미분방정식의 해석해와 수치해

미분방정식과 그 해가 무엇을 주는지부터, 가장 단순한 장난감 예제로 봅니다. 어떤 모델에서 헤지 논리를 적용했더니 다음 방정식이 나왔다고 **가정**합시다.

 $$\frac{dy}{dx} = y' = f'(x) \quad \quad f'(x) = \frac{1}{2}x^{-1/2} \quad \quad f(0) = 1$$

- 이 가상 모델에서는 옵션 가격이 주가 $x$ 하나에만 의존합니다. 실제 모델은 시간, 변동성 등 입력이 더 많습니다.
- 우리가 찾는 건 함수 $f(x)$ 자체입니다. 현재 주가 $S_t$ 를 $x$ 에 넣으면 옵션 가격이 나오는 **가격함수**입니다.
- $f(0) = 1$ 은 "주가가 0일 때 옵션 가격은 1"이라는 **조건**입니다. 미분방정식(기울기 정보)만으로는 해가 하나로 정해지지 않아서(아래 오른쪽 그림) 이런 조건이 꼭 필요합니다. BS에서는 만기 payoff와 경계조건이 이 역할을 합니다.
- 이 예제는 적분으로 **해석적으로**(공식으로) 풀 수 있습니다.

> ⚠️ $f'(x) = \frac12 x^{-1/2}$ 는 실제 가격결정 방정식이 아니라 개념 설명용 장난감입니다.

**아래 그림:** 왼쪽은 방정식이 주는 정보(각 주가에서의 기울기 $f'(x)$)이고, 오른쪽은 그 기울기를 가진 함수들입니다.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Generate x values (using positive values since we're taking square root)
x = np.linspace(0.01, 4, 200)  # Start from small positive number to avoid division by zero

# Create subplots
fig = make_subplots(rows=1, cols=2, subplot_titles=('Pricing Differential Equation: f\'(x) = 1/(2√x)', 'Option Pricing Function: f(x) = √x + C'))

# Plot derivative on left subplot
derivative = 1/(2*np.sqrt(x))
fig.add_trace(
    go.Scatter(
        x=x,
        y=derivative,
        mode='lines',
        line=dict(color='rgb(255, 165, 0)', width=2),
        showlegend=False
    ),
    row=1, col=1
)

# Plot different solutions for various values of C on right subplot
C_values = [-3, -2, -1, 0, 1, 2, 3]
base_opacity = 1
opacity_step = 0.8 / len(C_values)

# First add the general solution trace that will appear in legend
fig.add_trace(
    go.Scatter(
        x=x,
        y=np.sqrt(x) + C_values[0],
        mode='lines',
        line=dict(color='rgb(0, 191, 255)', width=2),
        name='General Solution'
    ),
    row=1, col=2
)

# Then add all other general solutions without showing in legend
for i, C in enumerate(C_values[1:]):
    if C != 1:  # Skip C=1 as it will be the particular solution
        opacity = base_opacity - (i * opacity_step)
        fig.add_trace(
            go.Scatter(
                x=x, 
                y=np.sqrt(x) + C,
                mode='lines',
                line=dict(
                    color='rgba(0, 191, 255, {})'.format(opacity),
                    width=2
                ),
                showlegend=False
            ),
            row=1, col=2
        )

# Add particular solution last so it appears on top
fig.add_trace(
    go.Scatter(
        x=x,
        y=np.sqrt(x) + 1,
        mode='lines',
        line=dict(
            color='rgb(57, 255, 20)',  # Neon green
            width=3
        ),
        name='Particular Solution'
    ),
    row=1, col=2
)

# Update layout
fig.update_layout(
    width=1200,  # Increased width for two subplots
    height=500,
    title_text='Option Pricing Differential Equation and Its Solutions',
    title_x=0.5,
    title_font_size=20,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

# Update axes
for i in [1, 2]:
    fig.update_xaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='rgba(128,128,128,0.2)',
        zeroline=True, 
        zerolinewidth=1, 
        zerolinecolor='rgba(128,128,128,0.5)',
        title_text='x - Stock Price',
        row=1, col=i
    )
    fig.update_yaxes(
        showgrid=True, 
        gridwidth=1, 
        gridcolor='rgba(128,128,128,0.2)',
        zeroline=True, 
        zerolinewidth=1, 
        zerolinecolor='rgba(128,128,128,0.5)',
        title_text='f(x) - Cost of Option in Dollars',
        row=1, col=i
    )

fig.show()


적분으로 풀면

$$\int f'(x)\,dx = \int \frac{1}{2}x^{- \frac{1}{2}}\,dx \implies f(x) = x^{\frac{1}{2}} + C$$
$$f(0) = 1 \implies 0^{\frac{1}{2}} + C = 1 \implies C = 1$$

따라서 가격함수는 $f(x) = \sqrt{x} + 1$ 이고, 주가 $x$ 를 넣으면 옵션 가격이 나옵니다.

**위 두 그림의 관계:** 기울기가 같은 곡선은 위아래로 평행이동한 것들(파란 선들, $+C$)이 모두 해입니다. 조건 $f(0)=1$ 이 그중 하나(초록 선)를 골라냅니다. **방정식은 모양을, 조건은 위치를 정합니다.**

**해석해가 좋은 이유:** 이 모델이 맞다고 가정하면, 공식은 **어떤 주가를 넣어도** 즉시 정확한 모델 가격을 줍니다. 근사도 없고 계산도 거의 필요 없습니다. 이런 해를 **해석해** 또는 **닫힌 형태(closed-form) 해**라고 부릅니다.

**다음 그림:** 해석해(왼쪽) 옆에 시장의 내재변동성 곡면(오른쪽)을 놓고, 공식이 있는 모델만으로는 왜 부족한지 봅니다.

In [ ]:
# Create subplots
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'xy'}, {'type': 'surface'}]],
    subplot_titles=('Option Pricing Function', 'Market Implied Volatility Surface')
)

# Generate x values for pricing function
x = np.linspace(0.01, 4, 200)

# Add the pricing function to first subplot
fig.add_trace(
    go.Scatter(
        x=x,
        y=np.sqrt(x) + 1,
        mode='lines',
        line=dict(
            color='rgb(57, 255, 20)',  # Neon green
            width=3
        ),
        name='Option Pricing Function',
        showlegend=False
    ),
    row=1, col=1
)

strikes = [90, 95, 100, 105, 110]
maturities = [1/12, 3/12, 6/12, 1, 2]    # in years
maturity_labels = ['1 Month', '3 Months', '6 Months', '1 Year', '2 Years']

# Market volatility values (in %)
market_vols = np.array([
    [28.0, 24.5, 22.0, 20.5, 19.5],  # 1 month
    [27.5, 24.0, 21.8, 20.3, 19.3],  # 3 months
    [27.0, 23.5, 21.5, 20.0, 19.0],  # 6 months
    [26.5, 23.0, 21.2, 19.8, 18.8],  # 1 year
    [26.0, 22.5, 21.0, 19.5, 18.5]   # 2 years
])

# Create meshgrid for 3D surface
X, Y = np.meshgrid(strikes, maturities)

# Add market volatility surface to second subplot
fig.add_trace(
    go.Surface(
        x=X,
        y=Y,
        z=market_vols,
        colorscale='Viridis',
        opacity=0.7,
        showscale=True,
        name='Market Volatility Surface',
        showlegend=False
    ),
    row=1, col=2
)

# Update layout
fig.update_layout(
    width=1000,
    height=600,
    title_x=0.5,
    title_font_size=20,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white')
)

# Update axes for pricing function plot
fig.update_xaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True, 
    zerolinewidth=1, 
    zerolinecolor='rgba(128,128,128,0.5)',
    title_text='x - Stock Price',
    row=1, col=1
)
fig.update_yaxes(
    showgrid=True, 
    gridwidth=1, 
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True, 
    zerolinewidth=1, 
    zerolinecolor='rgba(128,128,128,0.5)',
    title_text='f(x) - Cost of Option in Dollars',
    row=1, col=1
)

# Update 3D scene for volatility surface
fig.update_scenes(
    xaxis_title='Strike Price',
    yaxis_title='Time to Maturity (Years)',
    zaxis_title='Implied Volatility (%)',
    xaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
    yaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
    zaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
    bgcolor='rgba(0,0,0,0)',
    camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
)

fig.show()


**오른쪽 그림이 말하는 것:** 이것은 예시용으로 만든 시장 내재변동성(IV) 곡면입니다. BS 모델이 맞다면 IV는 행사가·만기와 상관없이 **하나의 평평한 면**이어야 합니다. 실제 시장에서는 행사가가 낮을수록 IV가 높은 **스큐**와 만기별 차이가 나타납니다. 즉 상수 σ를 가정하는 BS 모델은 시장 가격들을 한꺼번에 설명하지 못합니다.

**해석해가 가장 좋지만, 늘 가능한 건 아닙니다.** 그런데도 공식이 없는 모델을 쓰는 이유는, 그런 모델이 공식이 있는 모델은 담지 못하는 **시장의 중요한 특징**을 담기 때문입니다. 예를 들어 변동성이 상수가 아니라 무작위로 움직인다는 것(확률변동성), 그리고 위 그림 같은 IV 곡면의 모양과 움직임이 그렇습니다.

**그럼 공식이 없으면 어떻게 하나?** 가격결정 PDE의 해를 **근사**하는 여러 수치기법이 있고, 실무에 충분할 만큼 정확합니다. 다음 장부터 그 가장 기본적인 방법을 배웁니다.

---

##### 2.) 🌊 상미분방정식(ODE) 근사하기

**🎯 이 섹션의 목적:** 유한차분의 핵심 아이디어를 가장 단순한 1차원 문제에서 익힙니다. 여기서 배우는 **"미분을 차분으로 바꾸고, 한 칸씩 전진한다"**가 3장(열방정식)과 4장(BS)에서 그대로 확장됩니다.

공식(해석해)이 없으면 미분방정식을 **수치적으로** 근사할 수 있습니다. 컴퓨터는 단순한 계산을 엄청나게 많이 반복하는 데 강해서 이 방법과 잘 맞습니다. 왜 근사가 통하는지, 가장 간단한 **상미분방정식**(ODE, 변수가 하나인 미분방정식)으로 봅시다.

##### 📐 미분의 정의

함수가 미분 가능하면 도함수는 다음과 같이 정의됩니다.

$$f'(x) = \lim_{h \rightarrow 0} \frac{f(x+h) - f(x)}{h}$$

뜻: $x$ 와 거기서 $h$ 만큼 떨어진 점을 잇는 직선(할선)의 기울기를 구하고, $h$ 를 0으로 보냅니다. 그 극한이 **접선의 기울기**입니다.

학교에서는 "접선의 기울기"로 배우고 이 정의는 당연하게 넘기거나 귀찮아하는 경우가 많습니다. 하지만 수치해법에서는 이 정의가 **가장 강력한 도구**입니다.

**아래 그림:** $f(x) = x^2$ 와 $x = 1$ 에서의 접선 (기울기 $f'(1) = 2$).

In [ ]:
import numpy as np
import plotly.graph_objects as go

# Create a simple function and its derivative
x = np.linspace(-2, 2, 100)
f = x**2  # Example function: f(x) = x^2
f_prime = 2*x  # True derivative: f'(x) = 2x

# Point where we want to show the tangent line
x0 = 1
y0 = x0**2
slope = 2*x0

# Create points for tangent line with extended domain
x_tangent = np.array([x0-2, x0+2])  # Extended from [-0.5,0.5] to [-2,2]
y_tangent = slope*(x_tangent - x0) + y0

# Create the plot
fig = go.Figure()

# Plot the original function
fig.add_trace(go.Scatter(
    x=x, y=f,
    mode='lines',
    name='f(x) = x²',
    line=dict(color='rgb(0, 191, 255)', width=2),
    showlegend=True
))

# Plot the tangent line
fig.add_trace(go.Scatter(
    x=x_tangent, y=y_tangent,
    mode='lines',
    name=f'Tangent at x={x0}',
    line=dict(color='red', width=2, dash='dot'),
    showlegend=True
))

# Update layout
fig.update_layout(
    width=800,
    height=500,
    title_text='Function and its Tangent Line',
    title_x=0.5,
    title_font_size=20,
    plot_bgcolor='rgba(0,0,0,0)',
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    showlegend=True,
    legend=dict(
        yanchor="top",
        y=0.99,
        xanchor="right",
        x=0.99
    )
)

# Update axes
fig.update_xaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)',
    title_text='x'
)
fig.update_yaxes(
    showgrid=True,
    gridwidth=1,
    gridcolor='rgba(128,128,128,0.2)',
    zeroline=True,
    zerolinewidth=1,
    zerolinecolor='rgba(128,128,128,0.5)',
    title_text='y'
)

fig.show()


대학 미적분 수업에서는 이 극한 정의를 주로 미분 공식을 **증명**하는 데 씁니다. 예를 들어 멱함수 미분 공식:

**멱함수 미분 공식 증명:**

 $$\begin{align}
 \frac{d}{dx}x^n &= \lim_{h \to 0} \frac{(x+h)^n - x^n}{h} \\
 &= \lim_{h \to 0} \frac{\sum_{k=0}^n \binom{n}{k}x^{n-k}h^k - x^n}{h} \\
 &= \lim_{h \to 0} \frac{x^n + nx^{n-1}h + \frac{n(n-1)}{2}x^{n-2}h^2 + ... - x^n}{h} \\
 &= \lim_{h \to 0} \left(nx^{n-1} + \frac{n(n-1)}{2}x^{n-2}h + ...\right) \\
 &= nx^{n-1}
 \end{align}$$

**공식 사용:** $f(x)=x^2 \implies f'(x) = 2x$

이렇게만 배우면 정의가 왜 중요한지 와닿지 않습니다. 정의의 진짜 힘은 **근사**에 있습니다.

**미분 근사하기:** $f(x)$ 가 미분 가능하다면, $h$ 를 무한히 작게 보내는 대신 **그냥 아주 작은 숫자**로 정하면 됩니다.

$$f'(x) \approx \frac{f(x + h) - f(x)}{h} \quad \quad (h \text{ 가 작을 때})$$

**왜 이게 중요한가:** 컴퓨터는 극한을 계산할 수 없지만 뺄셈과 나눗셈은 할 수 있습니다. 이 식 하나로 **값만 계산할 수 있으면 어떤 함수든** 미분을 근사할 수 있습니다. 미분을 근사할 수 있으면 미분방정식도 근사할 수 있습니다.

아래 코드는 $x = 1$ 에서 근사값과 정확한 값을 비교합니다.

In [ ]:
# Function
def f(x):
    return x**2

# Actual Derivative
def f_prime(x):
    return 2*x

h = .0000001

print("Approximation:", (f(1 + h) - f(1))/h)
print("Actual:", f_prime(1))

(위 결과: 오차가 $10^{-7}$ 수준입니다. $\frac{(1+h)^2 - 1}{h} = 2 + h$ 라서 정확히 $h = 10^{-7}$ 만큼 틀리고, 소수점 끝자리의 나머지는 컴퓨터의 반올림 오차입니다.)

#####  💭 핵심 아이디어: 근사식을 뒤집으면?

$$f'(x) \approx \frac{f(x + h) - f(x)}{h}$$

$$\implies hf'(x) \approx f(x+h) - f(x)$$

$$\implies f(x+h) \approx f(x) + hf'(x)$$

**방금 한 일:** 원래는 "함수 → 미분" 방향이었는데, 뒤집으니 **"다음 값 ≈ 지금 값 + 기울기 × 한 걸음"**이 되었습니다. 원래 함수를 **도함수로 (근사적으로) 표현**한 것입니다.

**왜 이게 중요한가:** 미분방정식은 도함수(기울기)를 알려주는 식입니다. 그러니 **시작점 값 하나만 알면**, 기울기를 따라 한 걸음씩 나아가며 함수 전체를 반복적으로 복원할 수 있습니다.

이것이 ODE를 근사하는 **오일러 방법**이고, **유한차분법**의 전부입니다.

> 📌 **수정:** 원본 애니메이션은 제목은 "도함수로 함수를 복원"이었지만 코드가 매 스텝 **정답 $f(x)$ 를 그대로 찍고** 있었습니다(`y_current = f(x_current)`). 아래는 **도함수만** 써서 진짜로 복원하는 버전입니다.
>
> 규칙은 한 줄입니다: $y_{n+1} = y_n + h\cdot f'(x_n)$. 흰 선분이 "지금 점에서 기울기를 따라 $h$ 만큼 가는" 한 스텝이고, 빨간 점선이 누적 오차입니다.
> $f(x)=x^2$, $f'(x)=2x$ 에 일부러 큰 $h=0.25$ 를 쓰면 복원값은 정확히 $x_n^2 - h x_n$ 이 되어 오차가 $h\cdot x$ 로 쌓이는 게 보입니다. $h$ 를 줄이면 오차도 비례해서 줄어듭니다(1차 정확도, $O(h)$).

In [ ]:
import numpy as np
import plotly.graph_objects as go

def animate_euler(f_exact, f_prime, x0, y0, h, n_steps, slope_at="left", title=""):
    # Rebuild f from its derivative ONLY:  y_{n+1} = y_n + h * slope
    #   slope_at="left"  -> slope f'(x_n)      (forward Euler)
    #   slope_at="right" -> slope f'(x_{n+1})  (used when f'(x_0) is infinite)
    # f_exact is used only to draw the faint target curve and the error bars.
    xs = x0 + h * np.arange(n_steps + 1)
    ys = np.empty(n_steps + 1)
    ys[0] = y0
    slopes = np.array([f_prime(xs[n] if slope_at == "left" else xs[n + 1]) for n in range(n_steps)])
    for n in range(n_steps):
        ys[n + 1] = ys[n] + h * slopes[n]

    x_dense = np.linspace(x0, xs[-1], 400)
    def step_segment(n):
        if n >= n_steps:
            return [], []
        return [xs[n], xs[n + 1]], [ys[n], ys[n] + h * slopes[n]]
    def error_bar(n):
        return [xs[n], xs[n]], [ys[n], f_exact(xs[n])]
    def caption(n):
        return (f"{title}<br><sup>step {n}/{n_steps}:  x = {xs[n]:.2f}   rebuilt y = {ys[n]:.4f}   "
                f"true f(x) = {f_exact(xs[n]):.4f}   error = {ys[n] - f_exact(xs[n]):+.4f}</sup>")

    fig = go.Figure([
        go.Scatter(x=x_dense, y=f_exact(x_dense), mode='lines', line=dict(color='rgba(57, 255, 20, 0.35)', width=4),
                   name='true f(x) (only for comparison)'),
        go.Scatter(x=xs[:1], y=ys[:1], mode='lines+markers', line=dict(color='rgb(0, 191, 255)', width=3),
                   marker=dict(size=8), name='rebuilt from f\' only'),
        go.Scatter(x=step_segment(0)[0], y=step_segment(0)[1], mode='lines', line=dict(color='white', width=3),
                   name='next step: slope × h'),
        go.Scatter(x=error_bar(0)[0], y=error_bar(0)[1], mode='lines', line=dict(color='rgb(255, 75, 75)', width=2, dash='dot'),
                   name='error'),
    ])
    frames = []
    for n in range(n_steps + 1):
        seg, err = step_segment(n), error_bar(n)
        frames.append(go.Frame(
            data=[go.Scatter(x=xs[:n + 1], y=ys[:n + 1]), go.Scatter(x=seg[0], y=seg[1]), go.Scatter(x=err[0], y=err[1])],
            traces=[1, 2, 3], name=str(n), layout=go.Layout(title_text=caption(n))))
    fig.frames = frames
    fig.update_layout(
        width=900, height=550, title_text=caption(0), title_x=0.5,
        plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
        legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01, bgcolor='rgba(0,0,0,0)'),
        updatemenus=[{'type': 'buttons', 'showactive': False, 'x': 0.0, 'y': -0.12, 'xanchor': 'left', 'direction': 'left',
                      'buttons': [{'label': '▶️ Play', 'method': 'animate',
                                   'args': [None, {'frame': {'duration': 400, 'redraw': True}, 'fromcurrent': True}]},
                                  {'label': '⏸ Pause', 'method': 'animate',
                                   'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]}]}],
        sliders=[{'x': 0.18, 'len': 0.82, 'y': -0.05, 'currentvalue': {'prefix': 'step: '},
                  'steps': [{'label': str(n), 'method': 'animate',
                             'args': [[str(n)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}]}
                            for n in range(n_steps + 1)]}],
    )
    grid = dict(showgrid=True, gridwidth=1, gridcolor='rgba(128,128,128,0.2)', zeroline=True,
                zerolinewidth=1, zerolinecolor='rgba(128,128,128,0.5)')
    fig.update_xaxes(grid, title_text='x')
    fig.update_yaxes(grid, title_text='f(x)')
    return fig, xs, ys

fig, xs, ys = animate_euler(lambda x: x**2, lambda x: 2 * x, x0=0.0, y0=0.0, h=0.25, n_steps=20,
                            title="Rebuilding f(x) = x² from f'(x) = 2x only (forward Euler, h = 0.25)")
fig.show()
print(f"Rebuilt f(5) = {ys[-1]:.4f},  true f(5) = 25,  error = {ys[-1] - 25:+.4f}  (= -h·x = -0.25·5)")

**주의:** 수치해법은 **실제로 걸어간 구간**에서만 답을 줍니다. 계산하지 않은 구간의 값은 알 수 없어서, 알고 싶은 영역 전체를 한 걸음씩 차례로 채워야 합니다. 그리고 걸음마다 작은 오차가 생겨 **누적**됩니다(위 애니메이션의 빨간 점선). 걸음(h)을 작게 할수록 오차는 줄지만 계산은 늘어납니다. 이 **정확도와 계산량의 교환**이 수치해법 전반에 걸친 주제입니다.

###### ______________________________________________________________________________________________________________________________________


##### 유한차분 (Finite Differences)

유한차분법은 이게 **전부**입니다. **미분을 차분으로 근사하고, 식을 정리하고, 한 칸씩 반복해서 원래 함수를 구합니다.**

미분을 근사하는 방법은 어느 점들을 쓰느냐에 따라 여러 가지입니다.

 **전진차분** (지금 점과 **앞** 점):
$$\frac{\partial f}{\partial x} \approx \frac{f(x + \Delta x) - f(x)}{\Delta x} + O(\Delta x)$$

 **후진차분** (지금 점과 **뒤** 점):
 $$\frac{\partial f}{\partial x} \approx \frac{f(x) - f(x - \Delta x)}{\Delta x} + O(\Delta x)$$

 **중앙차분** (앞뒤 점을 대칭으로):
 $$\frac{\partial f}{\partial x} \approx \frac{f(x + \Delta x) - f(x - \Delta x)}{2\Delta x} + O(\Delta x^2)$$

 **2계 미분 (중앙차분)**: 곡률. BS의 감마 항, 열방정식의 확산 항에 필요합니다.
 $$\frac{\partial^2 f}{\partial x^2} \approx \frac{f(x + \Delta x) - 2f(x) + f(x - \Delta x)}{\Delta x^2} + O(\Delta x^2)$$

**$O(\Delta x)$, $O(\Delta x^2)$ 의 뜻 (정확도의 차수):** 간격 $\Delta x$ 를 반으로 줄이면 오차가 $O(\Delta x)$ 는 약 1/2로, $O(\Delta x^2)$ 는 약 1/4로 줄어듭니다. 테일러 전개 $f(x\pm\Delta x) = f \pm f'\Delta x + \tfrac12 f''\Delta x^2 \pm \dots$ 에서 두 식을 빼면 $\Delta x^2$ 항이 상쇄되기 때문에 중앙차분이 한 차수 더 정확합니다.

**왜 여러 종류가 필요한가:** 4장의 BS 방정식에는 $\partial V/\partial t$, $\partial V/\partial S$, $\partial^2V/\partial S^2$ 가 모두 나옵니다. 시간 방향은 한쪽 방향 차분(이미 아는 쪽의 값을 써야 하므로), 주가 방향은 정확한 중앙차분을 씁니다.

**요점:** 미분이나 미분방정식이 나와도 겁먹을 필요가 없습니다. 공식으로 풀지 못하더라도 이 방법으로 거의 항상 근사해를 구할 수 있습니다.

###### ______________________________________________________________________________________________________________________________________


##### 🔎 예제: 오일러 방법 (전진 유한차분)

**🎯 목적:** 1장의 장난감 가격결정 방정식을 이번에는 **공식 없이** 풀어봅니다. 정답($\sqrt x + 1$)을 알고 있으니 근사가 얼마나 맞는지 바로 확인할 수 있습니다.

  $$\frac{dy}{dx} = y' = f'(x) \quad \quad f'(x) = \frac{1}{2}x^{-1/2} \quad \quad f(0) = 1$$

**Step 1.** 방정식에서 출발합니다.
$$\frac{dy}{dx} = \frac{1}{2}x^{-1/2}$$

그리고 $x$ 축을 잘게 나눕니다(**이산화**).
- $x_n = nh$, $n = 0,1,\dots,N$ ($h$ = 한 걸음의 크기)
- $y_n$ 은 $y(x_n)$ 의 근사값

**Step 2.** 미분을 전진차분으로 근사합니다.
$$\frac{dy}{dx} \approx \frac{y_{n+1} - y_n}{h}$$

**Step 3.** 방정식에 대입합니다.
$$\frac{y_{n+1} - y_n}{h} = \frac{1}{2}x_n^{-1/2}$$

**Step 4.** 모르는 값 $y_{n+1}$ 에 대해 풉니다.
$$y_{n+1} = y_n + h\left(\frac{1}{2}x_n^{-1/2}\right)$$

**Step 5.** 구현 순서
1. 초기조건 $y_0 = 1$ 에서 시작
2. 작은 걸음 $h$ 선택
3. 각 $n$ 마다 $x_n = nh$ 를 계산하고, 위 식으로 $y_{n+1}$ 을 구하고, 다음 걸음으로

> ⚠️ **주의:** 이 예제는 $x_0 = 0$ 에서 $f'(0) = \frac{1}{2\sqrt{0}} = \infty$ 라서, 위의 순수 전진 오일러($x_n$ 의 기울기 사용)로는 **첫 걸음부터 계산이 불가능**합니다.
> 그래서 아래 코드는 각 걸음의 **오른쪽 끝** 기울기를 씁니다: $y_{n+1} = y_n + h\,f'(x_{n+1})$.
> 도함수가 $x$ 만의 함수일 때 이 방식은 $\int f'(x)dx$ 를 오른쪽 리만합으로 근사하는 것과 같습니다.
> 그리고 $x=0$ 근처의 무한대 기울기 때문에 오차가 보통의 $O(h)$ 보다 느리게($\approx O(\sqrt h)$) 줄어듭니다. 아래 표에서 확인합니다.

###### ______________________________________________________________________________________________________________________________________


##### 💻 코드로 풀기: ODE 유한차분

아래 셀은 위 Step 5를 그대로 코드로 옮긴 것입니다. `euler_solve` 함수 하나로 두 방식(왼쪽 끝 기울기 / 오른쪽 끝 기울기)을 모두 다룹니다. 그 다음 셀에서 정답과 비교하고, 걸음 $h$ 를 줄이면 오차가 어떻게 줄어드는지 확인합니다.

In [ ]:
def euler_solve(f_prime, x0, y0, x_end, n_steps, slope_at="left"):
    # y_{n+1} = y_n + h * f'(x_n)       if slope_at == "left"  (forward Euler)
    # y_{n+1} = y_n + h * f'(x_{n+1})   if slope_at == "right" (needed here: f'(0) is infinite)
    x = np.linspace(x0, x_end, n_steps + 1)
    h = x[1] - x[0]
    y = np.empty_like(x)
    y[0] = y0
    for n in range(n_steps):
        y[n + 1] = y[n] + h * f_prime(x[n] if slope_at == "left" else x[n + 1])
    return x, y

f_prime = lambda x: 0.5 * x ** (-0.5)
f_exact = lambda x: np.sqrt(x) + 1

x_d, y_hat = euler_solve(f_prime, x0=0.0, y0=1.0, x_end=5.0, n_steps=2000, slope_at="right")

In [ ]:
print(f"Euler (2,000 steps):  y(5) = {y_hat[-1]:.5f}   exact √5 + 1 = {f_exact(5):.5f}   error = {y_hat[-1] - f_exact(5):+.5f}\n")

# How fast does the error shrink as the step h shrinks?
rows = []
for n_steps in (20, 200, 2_000, 20_000):
    _, y_sing = euler_solve(f_prime, 0.0, 1.0, 5.0, n_steps, slope_at="right")   # starts at the singular point x = 0
    _, y_smooth = euler_solve(f_prime, 1.0, 2.0, 5.0, n_steps, slope_at="left")  # starts at x = 1 where f' is smooth
    rows.append((n_steps, 5 / n_steps, y_sing[-1] - f_exact(5), y_smooth[-1] - f_exact(5)))

print(f"{'steps':>7} {'h (from 0)':>11} {'error: start x=0 (right slope)':>32} {'error: start x=1 (forward Euler)':>34}")
for n, h, e1, e2 in rows:
    print(f"{n:>7,} {h:>11.5f} {e1:>32.6f} {e2:>34.6f}")
print("\n→ Starting at x = 1 (smooth slope): error ÷10 when h ÷10  → first order, O(h).")
print("→ Starting at x = 0 (infinite slope): error only ÷√10 ≈ 3.2 when h ÷10 → the singularity dominates, O(√h).")

###### ______________________________________________________________________________________________________________________________________


##### 🖼️ 수치해와 해석해를 눈으로 비교하기

> 📌 **수정:** 원본 애니메이션도 정답 $\sqrt{x}+1$ 을 찍고 있었습니다. 아래는 위 코드와 **같은 방식**(오른쪽 끝 기울기, $x_0 = 0$, $y_0 = 1$)으로 도함수만 써서 복원합니다. 눈에 보이도록 $h = 0.25$ 로 크게 잡았습니다.
> $f'$ 가 감소함수라 오른쪽 끝 기울기는 항상 구간 평균보다 작고, 그래서 복원값이 정답보다 계속 아래에 있습니다. 첫 스텝(기울기가 폭발하는 $x=0$ 근처)에서 생긴 오차가 가장 큽니다.

In [ ]:
fig, xs, ys = animate_euler(lambda x: np.sqrt(x) + 1, lambda x: 0.5 * x ** (-0.5), x0=0.0, y0=1.0, h=0.25, n_steps=20,
                            slope_at="right", title="Rebuilding f(x) = √x + 1 from f'(x) = 1/(2√x) only (h = 0.25)")
fig.update_yaxes(range=[0, 4])
fig.show()

**아래 그림:** 걸음 수를 세 가지로 바꿔 수치해를 해석해와 겹쳐 봅니다. 오른쪽은 오차(수치해 − 해석해)입니다.
- 걸음이 작을수록(걸음 수가 많을수록) 해석해에 가까워집니다.
- 오차 곡선이 $x=0$ 근처에서 급히 떨어지고 그 뒤로는 거의 평평합니다. **오차 대부분이 기울기가 폭발하는 첫 몇 걸음에서 생긴다**는 뜻입니다. 방정식이 까다로운 곳에서는 격자를 더 촘촘하게 해야 한다는 교훈입니다.

In [ ]:
# Numerical vs analytical for several step sizes
fig = make_subplots(rows=1, cols=2, subplot_titles=('Euler solutions vs. analytical f(x) = √x + 1', 'Error  (Euler − analytical)'))
x_dense = np.linspace(0, 5, 500)
fig.add_trace(go.Scatter(x=x_dense, y=f_exact(x_dense), mode='lines', line=dict(color='rgb(57, 255, 20)', width=4),
                         name='analytical √x + 1'), row=1, col=1)
for n_steps, color in [(10, 'rgb(255, 75, 75)'), (50, 'rgb(255, 165, 0)'), (2000, 'rgb(0, 191, 255)')]:
    xe, ye = euler_solve(f_prime, 0.0, 1.0, 5.0, n_steps, slope_at="right")
    fig.add_trace(go.Scatter(x=xe, y=ye, mode='lines+markers' if n_steps <= 50 else 'lines', marker=dict(size=5),
                             line=dict(color=color, width=2), name=f'Euler, {n_steps} steps (h = {5 / n_steps:g})'), row=1, col=1)
    fig.add_trace(go.Scatter(x=xe, y=ye - f_exact(xe), mode='lines', line=dict(color=color, width=2), showlegend=False), row=1, col=2)
fig.update_layout(width=1200, height=500, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(yanchor="bottom", y=0.02, xanchor="right", x=0.45, bgcolor='rgba(0,0,0,0)'))
for col in (1, 2):
    fig.update_xaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', title_text='x - Stock Price', row=1, col=col)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.update_yaxes(title_text='f(x) - Cost of Option in Dollars', row=1, col=1)
fig.update_yaxes(title_text='error', row=1, col=2)
fig.show()

---

##### 3.) 🔥 편미분방정식(PDE) 근사하기

**🎯 이 섹션의 목적:** 변수가 두 개($x$ 와 $t$)인 편미분방정식으로 넘어갑니다. 아이디어는 2장과 같지만 두 가지가 새로 나옵니다.
1. 선이 아니라 **격자(표)**를 채운다.
2. 간격을 아무렇게나 잡으면 결과가 폭발한다: **안정성** 문제.

BS PDE가 바로 이런 PDE라서, 여기서 익힌 것이 4장에서 그대로 쓰입니다.

같은 방법을 편미분방정식에 적용합니다. 예제는 1차원 **열방정식**입니다.

*왜 하필 열방정식인가?* BS PDE는 수학적으로 **2계 포물형(parabolic) PDE**이고, 변수를 바꾸면($x = \ln S$, $\tau = T - t$ 등) 정확히 열방정식이 됩니다. 옵션 가격이 만기에서 멀어질수록 매끈해지는 현상은 열이 퍼지는 것과 같습니다.

 $$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$$

 - $u(x,t)$: 위치 $x$, 시간 $t$ 에서의 온도
 - $\alpha$: 열확산 계수 (열이 얼마나 빨리 퍼지나)
 - $\frac{\partial u}{\partial t}$: 온도가 시간에 따라 변하는 속도
 - $\frac{\partial^2 u}{\partial x^2}$: 위치에 따른 온도 분포의 곡률(2계 미분)

**직관:** 어떤 점이 주변보다 차가우면(아래로 볼록, $u_{xx} > 0$) 데워지고, 주변보다 뜨거우면(위로 볼록) 식습니다. 즉 **울퉁불퉁한 분포가 점점 평평해집니다.** 이 방정식은 1차원 막대에서 열이 퍼지는 과정을 나타냅니다.

**아래 그림:** 막대 양 끝을 0도로 유지하고 처음 온도 분포가 $\sin(\pi x)$ 인 경우의 정확한 해 $u = e^{-\pi^2 t}\sin(\pi x)$ 입니다. 시간이 갈수록 모양은 그대로 둔 채 전체가 식어갑니다. 다음 셀들에서 이것을 공식 없이 격자로 계산해 비교합니다.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Generate x and t values for 3D surface
x = np.linspace(0, 1, 100)
t = np.linspace(0, 1, 200)  # Increased number of points while keeping same interval
X, T = np.meshgrid(x, t)

# Calculate u(x,t) = e^(-π²t) * sin(πx)
Z = np.exp(-np.pi**2 * T) * np.sin(np.pi * X)

# Create figure with subplots
fig = make_subplots(
    rows=1, cols=1,
    specs=[[{'type': 'surface'}]]
)

# Add surface plot
fig.add_trace(
    go.Surface(
        x=X,
        y=T,
        z=Z,
        colorscale='Viridis',
        opacity=0.7,
        showscale=True
    ),
    row=1, col=1
)

# Update layout
fig.update_layout(
    title='Heat Equation Solution Surface Plot',
    width=1000,
    height=600,
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white'),
    scene=dict(
        xaxis_title='x - Position',
        yaxis_title='t - Time',
        zaxis_title='u(x,t) - Temperature',
        xaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
        yaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
        zaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
        bgcolor='rgba(0,0,0,0)',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.2))
    )
)

fig.show()


###### ______________________________________________________________________________________________________________________________________


##### 🔎 예제: 열방정식을 explicit 유한차분으로 풀기, 단계별로

$\frac{\partial u}{\partial t} = \alpha \frac{\partial^2 u}{\partial x^2}$ 를 explicit 유한차분으로 풉니다.

**Step 1.** 공간과 시간을 모두 잘게 나눕니다.
- 공간: $x_i = i\Delta x$, $i = 0,1,\dots,N$
- 시간: $t_j = j\Delta t$, $j = 0,1,\dots,M$
- $u^j_i$ 는 $u(x_i,t_j)$ 의 근사값. 행 = 시간, 열 = 위치인 **표**를 채운다고 생각하면 됩니다.

**Step 2.** 미분을 차분으로 근사합니다. 시간은 전진차분, 공간은 2계 중앙차분입니다.
$$\frac{\partial u}{\partial t} \approx \frac{u^{j+1}_i - u^j_i}{\Delta t}$$
$$\frac{\partial^2 u}{\partial x^2} \approx \frac{u^j_{i+1} - 2u^j_i + u^j_{i-1}}{(\Delta x)^2}$$

**Step 3.** 열방정식에 대입합니다.
$$\frac{u^{j+1}_i - u^j_i}{\Delta t} = \alpha \frac{u^j_{i+1} - 2u^j_i + u^j_{i-1}}{(\Delta x)^2}$$

**Step 4.** 다음 시점의 값 $u^{j+1}_i$ 에 대해 풉니다.
$$u^{j+1}_i = u^j_i + \alpha\frac{\Delta t}{(\Delta x)^2}(u^j_{i+1} - 2u^j_i + u^j_{i-1})$$

→ **다음 시점의 한 칸 = 지금 시점의 이웃 세 칸으로 계산.** 오른쪽 변은 전부 이미 아는 값이라 바로 계산됩니다. 이렇게 이미 아는 값만으로 바로 계산되는 방식을 **explicit(명시적) 방법**이라고 부릅니다.

**Step 5.** 구현 순서
1. 초기조건 $u(x,0)$ 로 첫 행을 채운다
2. 안정성 조건 $\alpha\frac{\Delta t}{(\Delta x)^2} \leq \frac{1}{2}$ 을 만족하도록 $\Delta x$, $\Delta t$ 를 고른다
3. 각 시간 스텝 $j$ 마다
   - 모든 내부 점 $i$ 의 $u^{j+1}_i$ 를 위 식으로 계산
   - 양 끝(경계조건)을 채움
   - 다음 시간 스텝으로

**핵심 개념: 안정성.** 안정성은 계산을 반복하는 동안 **오차가 커지느냐**의 문제입니다. 조건 $\alpha\frac{\Delta t}{(\Delta x)^2} \leq \frac{1}{2}$ 을 지키면 오차가 커지지 않습니다. 공간을 촘촘하게($\Delta x$ 작게) 하면 시간 간격은 **그 제곱에 비례해** 더 작게 잡아야 한다는 뜻입니다. 왜 하필 ½인지는 이 장 끝에서 직접 실험으로 확인합니다.

###### ______________________________________________________________________________________________________________________________________


##### 💻 코드로 풀기: PDE 유한차분

아래 코드는 Step 5를 그대로 옮긴 것입니다. `u[j, i]` 에서 `j` 가 시간(행), `i` 가 위치(열)입니다. 바깥 루프가 시간을 한 스텝씩 **앞으로** 진행하고, 안쪽 루프가 그 시점의 모든 내부 점을 채웁니다. 다음 셀에서 정확한 해와의 오차를 확인합니다.

In [ ]:
# Explicit finite differences for u_t = alpha * u_xx on x in [0, 1]
#   initial condition   u(x, 0) = sin(pi x)
#   boundary conditions u(0, t) = u(1, t) = 0
alpha = 1.0
L = 1.0
T_end = 1.0
dx = 0.1
dt = 0.005

lam = alpha * dt / dx ** 2          # the single number that decides stability
print(f"lambda = alpha·dt/dx² = {lam:.3f}  →  {'stable' if lam <= 0.5 else 'UNSTABLE'} (need lambda ≤ 0.5)")

N = int(round(L / dx)) + 1          # number of space points
M = int(round(T_end / dt)) + 1      # number of time points

x = np.linspace(0, L, N)
t = np.linspace(0, T_end, M)

u = np.zeros((M, N))                # row j = time t_j, column i = position x_i
u[0, :] = np.sin(np.pi * x)
u[:, 0] = u[:, -1] = 0

for j in range(0, M - 1):           # march FORWARD in time
    for i in range(1, N - 1):       # every interior point uses its two neighbours at the previous time
        u[j + 1, i] = u[j, i] + lam * (u[j, i + 1] - 2 * u[j, i] + u[j, i - 1])

In [ ]:
u_exact = np.exp(-np.pi ** 2 * t[:, None]) * np.sin(np.pi * x[None, :])
print(f"{'t':>6} {'max u (exact)':>15} {'max |numerical − exact|':>25}")
for tt in (0.0, 0.01, 0.1, 0.5, 1.0):
    j = int(np.argmin(np.abs(t - tt)))
    print(f"{t[j]:>6.2f} {u_exact[j].max():>15.3e} {np.abs(u[j] - u_exact[j]).max():>25.3e}")

###### ______________________________________________________________________________________________________________________________________


##### 🖼️ 수치해와 해석해를 눈으로 비교하기

첫 그림은 격자로 계산한 값(왼쪽)과 정확한 공식(오른쪽)입니다. 모양이 같으면 성공입니다.
그 다음 애니메이션은 시간이 흐르며 격자가 한 줄씩 채워지는 과정을 3D로 보여줍니다(진한 면: 수치해, 옅은 면: 해석해). ▶ Play를 누르세요.

In [ ]:
# Reuse the grid solved above
u_numeric = u
u_analytic = u_exact
X, T_grid = np.meshgrid(x, t)

fig = make_subplots(rows=1, cols=2, specs=[[{'type': 'surface'}, {'type': 'surface'}]],
                    subplot_titles=('Numerical Solution', 'Analytical Solution'))
fig.add_trace(go.Surface(x=X, y=T_grid, z=u_numeric, colorscale='Viridis', opacity=0.7, showscale=True), row=1, col=1)
fig.add_trace(go.Surface(x=X, y=T_grid, z=u_analytic, colorscale='Viridis', opacity=0.7, showscale=False), row=1, col=2)

scene = dict(
    xaxis_title='x - Position', yaxis_title='t - Time', zaxis_title='u(x,t) - Temperature',
    xaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
    yaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
    zaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
    bgcolor='rgba(0,0,0,0)', camera=dict(eye=dict(x=1.5, y=1.5, z=1.2)))
fig.update_layout(title='Heat Equation Solutions: Numerical vs Analytical', width=1500, height=600,
                  paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'), scene=scene, scene2=scene)
fig.show()

In [ ]:
# Create 3D animation showing how the numerical solution evolves over time
frames = []

# Create base figure with 3D axes
fig = go.Figure()

# Add frames for each time step
for i in range(0, M, 10):  # Sample every 10th time step
    # Create meshgrid for current time slice
    X_slice, T_slice = np.meshgrid(x, t[:i+1])
    
    frames.append(
        go.Frame(
            data=[
                go.Surface(
                    x=X_slice,
                    y=T_slice,
                    z=u_numeric[:i+1,:],
                    colorscale='Viridis',
                    opacity=0.8,
                    showscale=True,
                    name='Numerical Solution'
                ),
                go.Surface(
                    x=X_slice,
                    y=T_slice,
                    z=u_analytic[:i+1,:],
                    colorscale='Plasma',
                    opacity=0.4,
                    showscale=False,
                    name='Analytical Solution'
                )
            ],
            name=f't={t[i]:.3f}'
        )
    )

# Add initial surface
fig.add_trace(
    go.Surface(
        x=np.meshgrid(x, t[:1])[0],
        y=np.meshgrid(x, t[:1])[1],
        z=u_numeric[:1,:],
        colorscale='Viridis',
        opacity=0.8,
        showscale=True,
        name='Numerical Solution'
    )
)

fig.add_trace(
    go.Surface(
        x=np.meshgrid(x, t[:1])[0],
        y=np.meshgrid(x, t[:1])[1],
        z=u_analytic[:1,:],
        colorscale='Plasma',
        opacity=0.4,
        showscale=False,
        name='Analytical Solution'
    )
)

# Update layout
fig.update_layout(
    title='Evolution of Heat Equation Solution in 3D',
    width=1000,
    height=800,
    scene=dict(
        xaxis_title='Position (x)',
        yaxis_title='Time (t)',
        zaxis_title='Temperature u(x,t)',
        xaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
        yaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
        zaxis=dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)'),
        bgcolor='rgba(0,0,0,0)',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2),
            up=dict(x=0, y=0, z=1)
        )
    ),
    paper_bgcolor='rgba(0,0,0,0)',
    font=dict(color='white')
)

# Add frames and animation settings
fig.frames = frames
fig.update_layout(
    updatemenus=[
        dict(
            type='buttons',
            showactive=False,
            buttons=[
                dict(
                    label='Play',
                    method='animate',
                    args=[None, dict(
                        frame=dict(duration=100, redraw=True),
                        fromcurrent=True,
                        mode='immediate'
                    )]
                ),
                dict(
                    label='Pause',
                    method='animate',
                    args=[[None], dict(
                        frame=dict(duration=0, redraw=False),
                        mode='immediate',
                        transition=dict(duration=0)
                    )]
                )
            ],
            x=0.1,
            y=0
        )
    ]
)

fig.show()


###### ______________________________________________________________________________________________________________________________________

##### ⚖️ 안정성 조건 $\lambda \le \tfrac12$ 은 어디서 나오나

업데이트 식을 같은 칸끼리 묶으면

$$u_i^{j+1} = \lambda\,u_{i-1}^j + (1-2\lambda)\,u_i^j + \lambda\,u_{i+1}^j, \qquad \lambda = \frac{\alpha\Delta t}{\Delta x^2}$$

**다음 시점의 값 = 이웃 세 칸의 가중합**이고, 가중치의 합은 $\lambda + (1-2\lambda) + \lambda = 1$ 입니다.

- $\lambda \le \tfrac12$ 이면 가중치가 모두 $\ge 0$ 이라 새 값은 이웃들의 **가중평균**입니다. 평균은 원래 값들의 최댓값을 넘을 수 없으니 오차도 커질 수 없습니다.
- $\lambda > \tfrac12$ 이면 가운데 가중치 $1-2\lambda$ 가 **음수**가 됩니다. 그러면 칸마다 +/−가 번갈아 나타나는 톱니 모양 오차가 스텝마다 부호를 바꾸며 커집니다(한 스텝에 $|1-4\lambda|$ 배, $\lambda=0.6$ 이면 1.4배).

아래 셀은 $\lambda$ 만 바꿔서 확인합니다. 초기조건에 일부러 오차를 넣지 않았는데도 $\lambda=0.6$ 에서는 컴퓨터의 반올림 오차($10^{-16}$ 수준)가 톱니 모양으로 증폭돼 폭발합니다.

**이 "가중합, 가중치는 음수 금지" 구조가 BS 유한차분에서도 그대로 나옵니다.**

In [ ]:
def heat_explicit(lam, dx=0.1, T_end=1.0):
    dt = lam * dx ** 2
    xg = np.linspace(0, 1, int(round(1 / dx)) + 1)
    n_t = int(round(T_end / dt))
    ug = np.sin(np.pi * xg)
    history, max_abs = [ug.copy()], [np.abs(ug).max()]
    for _ in range(n_t):
        ug = ug.copy()
        ug[1:-1] = lam * ug[:-2] + (1 - 2 * lam) * ug[1:-1] + lam * ug[2:]   # weights: lam, 1-2lam, lam
        history.append(ug)
        max_abs.append(np.abs(ug).max())
    return xg, np.arange(n_t + 1) * dt, np.array(history), np.array(max_abs)

fig = make_subplots(rows=1, cols=2, subplot_titles=('max |u| over time (log scale)', 'λ = 0.6: u(x) at the last steps'))
t_dense = np.linspace(0, 1, 200)
fig.add_trace(go.Scatter(x=t_dense, y=np.exp(-np.pi ** 2 * t_dense), mode='lines', line=dict(color='white', width=3, dash='dash'),
                         name='exact e^(−π²t)'), row=1, col=1)
for lam, color in [(0.4, 'rgb(57, 255, 20)'), (0.5, 'rgb(0, 191, 255)'), (0.6, 'rgb(255, 75, 75)')]:
    xg, tg, hist, mx = heat_explicit(lam)
    fig.add_trace(go.Scatter(x=tg, y=mx, mode='lines', line=dict(color=color, width=2),
                             name=f'λ = {lam}  (center weight 1−2λ = {1 - 2 * lam:+.1f})'), row=1, col=1)
    print(f"λ = {lam}: max|u| at t = 1 → {mx[-1]:.3e}   (exact {np.exp(-np.pi ** 2):.3e})")
    if lam == 0.6:
        for k, shade in zip((-3, -2, -1), (0.35, 0.65, 1.0)):
            fig.add_trace(go.Scatter(x=xg, y=hist[k], mode='lines+markers', line=dict(color=f'rgba(255, 75, 75, {shade})'),
                                     name=f'λ = 0.6, t = {tg[k]:.3f}'), row=1, col=2)
fig.update_yaxes(type='log', exponentformat='power', title_text='max |u|', row=1, col=1)
fig.update_xaxes(title_text='t', row=1, col=1)
fig.update_xaxes(title_text='x', row=1, col=2)
fig.update_layout(width=1300, height=500, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(x=0.01, y=0.99, yanchor='top', bgcolor='rgba(0,0,0,0)'))
for col in (1, 2):
    fig.update_xaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.show()

---

##### 4.) 📝 Black-Scholes PDE 근사하기

**🎯 이 섹션의 목적:** 드디어 진짜 목표입니다. 앞에서 익힌 두 가지(미분을 차분으로 바꾸기, 격자를 한 칸씩 채우기)를 BS PDE에 적용해 옵션 가격을 계산하고, 공식(해석해)과 비교해 방법이 맞는지 **검증**합니다. 공식이 있는 BS에서 검증해두면, 공식이 없는 상품(이 장 마지막 셀의 아메리칸 옵션)에도 믿고 쓸 수 있습니다.

이제 진짜 가격결정 PDE를 풉니다 (배당 없는 유럽형 콜).

$$\frac{\partial V}{\partial t} + \frac{1}{2}\sigma^2S^2\frac{\partial^2 V}{\partial S^2} + rS\frac{\partial V}{\partial S} - rV = 0$$

열방정식과 결정적으로 다른 점이 하나 있습니다. **알고 있는 조건이 시작($t=0$)이 아니라 끝($t=T$)에 있습니다.**
- 만기의 옵션 가치는 payoff 그 자체입니다: $V(S,T) = \max(S-K, 0)$. 이게 **종료조건**입니다.
- 우리가 원하는 건 오늘의 가격 $V(S, 0)$ 입니다.

그래서 **만기에서 출발해 시간을 거꾸로** 계산합니다. (BS PDE는 $\tau = T-t$ 로 바꾸면 열방정식처럼 "앞으로 가는" 모양이 됩니다. 만기의 꺾인 payoff가 열처럼 퍼지며 매끈해지는 과정입니다.)

**Step 1. 격자(표) 만들기**
- 주가 축: $S_i = i\,\Delta S,\ i = 0,\dots,M,\ \Delta S = S_{max}/M$
- 시간 축: $t_j = j\,\Delta t,\ j = 0,\dots,N,\ \Delta t = T/N$
- $V_i^j \approx V(S_i, t_j)$: 이 $(M+1)\times(N+1)$ 표의 칸을 하나하나 채우는 게 목표입니다.

**Step 2. 처음부터 알고 있는 칸 (조건)**

| 위치 | 값 | 이유 |
|---|---|---|
| 오른쪽 끝 열 $t_N = T$ | $\max(S_i - K, 0)$ | 만기 payoff (종료조건) |
| 맨 아래 행 $S_0 = 0$ | $0$ | 주가가 0이면 GBM에서는 계속 0이라 콜은 무가치 |
| 맨 위 행 $S_M = S_{max}$ | $S_{max} - Ke^{-r(T-t_j)}$ | 아주 깊은 ITM 콜은 사실상 "주식 − 행사가의 현재가치" (행사가 거의 확실) |

> ⚠️ **원본 코드의 버그:** 맨 위 행을 설정하지 않아 $V(S_{max}, t) = 0$ 으로 남아 있었습니다. "주가가 150이면 콜 가치가 0"이라는 틀린 조건이 격자 안으로 퍼져서 $S_{max}=150$ 결과가 크게 틀렸습니다. 원본은 이것을 "$S_{max}$ 를 키워야 한다"로 설명했지만, 실제 원인은 경계조건입니다. 아래에서 수치로 확인합니다.

**Step 3. 미분을 차분으로 바꾸기.** 핵심은 공간 미분을 **이미 채워진 열 $j+1$** 에서 계산하는 것입니다 (explicit scheme).

$$\frac{\partial V}{\partial t} \approx \frac{V_i^{j+1} - V_i^j}{\Delta t},\qquad \frac{\partial V}{\partial S} \approx \frac{V^{j+1}_{i+1} - V^{j+1}_{i-1}}{2\Delta S},\qquad \frac{\partial^2 V}{\partial S^2} \approx \frac{V^{j+1}_{i+1} - 2V^{j+1}_i + V^{j+1}_{i-1}}{(\Delta S)^2}$$

**Step 4. 모르는 칸 $V_i^j$ 에 대해 풀기.** (원본 설명은 $V^{j+1}$ 을 푸는 방향으로 적혀 있었습니다. 코드는 맞게 $V^j$ 를 풀고 있었고, 설명만 거꾸로였습니다.)

$$V_i^j = V_i^{j+1} + \Delta t\left(\tfrac12\sigma^2S_i^2\,\frac{V^{j+1}_{i+1} - 2V^{j+1}_i + V^{j+1}_{i-1}}{(\Delta S)^2} + rS_i\,\frac{V^{j+1}_{i+1} - V^{j+1}_{i-1}}{2\Delta S} - rV_i^{j+1}\right)$$

$S_i / \Delta S = i$ 를 써서 같은 칸끼리 묶으면

$$\boxed{V_i^j = a_i\,V_{i-1}^{j+1} + b_i\,V_i^{j+1} + c_i\,V_{i+1}^{j+1}}$$

$$a_i = \tfrac12\Delta t\,(\sigma^2 i^2 - r i),\qquad b_i = 1 - \Delta t\,(\sigma^2 i^2 + r),\qquad c_i = \tfrac12\Delta t\,(\sigma^2 i^2 + r i)$$

**모르는 칸 하나 = 한 스텝 뒤 시점(오른쪽 열)에 있는 이웃 세 칸의 가중합.** 오른쪽 열은 이미 다 채워져 있으니, 한 열씩 왼쪽으로(과거로) 채워 나가면 됩니다. 마지막에 채워지는 왼쪽 끝 열이 오늘의 가격곡선 $V(S, 0)$ 입니다.

**Step 5. 안정성.** 열방정식과 마찬가지로 가중치가 모두 $\ge 0$ 이면 안전합니다(충분조건). 가장 빡빡한 조건은 $i = M$ 에서의 $b_M \ge 0$ 입니다.

$$\Delta t \le \frac{1}{\sigma^2 M^2 + r} \approx \frac{1}{\sigma^2 M^2}$$

$\Delta S$ 를 반으로 줄이면($M$ 두 배) $\Delta t$ 는 4분의 1로 줄여야 합니다. explicit 방법의 대표적인 비용입니다.

In [ ]:
import pandas as pd
from scipy.stats import norm

# Contract and model parameters used in the rest of this section
K = 100        # strike
T = 1.0        # time to maturity (years)
r = 0.05       # risk-free rate
sigma = 0.2    # volatility

def bs_explicit_fd(S_max, M, N, K=K, T=T, r=r, sigma=sigma, upper_bc="linear"):
    # Explicit finite differences for a European call, marching BACKWARD from the payoff.
    # Returns S (M+1), t (N+1), the full grid V[i, j] ≈ V(S_i, t_j), and the weights (a, b, c).
    # upper_bc = "linear": V(S_max, t) = S_max - K e^{-r(T-t)}   (correct)
    #            "zero"  : V(S_max, t) = 0                       (what the original notebook effectively did)
    dt = T / N
    S = np.linspace(0, S_max, M + 1)
    t = np.linspace(0, T, N + 1)
    i = np.arange(M + 1)                        # S_i / dS = i
    a = 0.5 * dt * (sigma**2 * i**2 - r * i)    # weight on the node one step DOWN
    b = 1 - dt * (sigma**2 * i**2 + r)          # weight on the SAME node
    c = 0.5 * dt * (sigma**2 * i**2 + r * i)    # weight on the node one step UP

    V = np.full((M + 1, N + 1), np.nan)         # NaN = not computed yet
    V[:, N] = np.maximum(S - K, 0)              # terminal condition: the payoff
    for j in range(N - 1, -1, -1):              # j = N-1, ..., 0  (backward in time)
        # every interior node of column j from its three neighbours in column j+1
        # (same as the double loop in the original code, written for all i at once)
        V[1:M, j] = a[1:M] * V[0:M - 1, j + 1] + b[1:M] * V[1:M, j + 1] + c[1:M] * V[2:M + 1, j + 1]
        V[0, j] = 0.0                                                            # S = 0
        V[M, j] = S_max - K * np.exp(-r * (T - t[j])) if upper_bc == "linear" else 0.0  # S = S_max
    return S, t, V, (a, b, c)

def bs_call_exact(S, K=K, tau=T, r=r, sigma=sigma):
    # Closed-form Black-Scholes call with time-to-maturity tau (handles S = 0 and tau = 0)
    S = np.asarray(S, dtype=float)
    if tau <= 0:
        return np.maximum(S - K, 0.0)
    out = np.zeros_like(S)
    pos = S > 0
    d1 = (np.log(S[pos] / K) + (r + 0.5 * sigma**2) * tau) / (sigma * np.sqrt(tau))
    d2 = d1 - sigma * np.sqrt(tau)
    out[pos] = S[pos] * norm.cdf(d1) - K * np.exp(-r * tau) * norm.cdf(d2)
    return out

print(f"Closed-form price at S = 100: {bs_call_exact(100.0):.4f}")

##### 원본 결과 다시 보기: 무엇이 틀렸고 어떻게 고치나

원본은 $M=100$, $N=1000$ 으로 $S_{max}=150$ 과 $S_{max}=2000$ 을 비교했습니다. 같은 설정으로 세 경우를 나란히 놓습니다.
1. **원본 그대로, $S_{max}=150$**: 위쪽 경계가 0으로 고정되어 있음
2. **원본 그대로, $S_{max}=2000$**: 경계가 멀어서 버그의 영향은 줄었지만, $\Delta S = 2000/100 = 20$ 이라 70~150 구간에 격자점이 **4개**뿐임
3. **경계조건만 고친 $S_{max}=150$**: 해석해와 거의 겹침

In [ ]:
cases = [
    ("original: S_max=150, upper BC = 0",  150, "zero",   'rgb(255, 165, 0)'),
    ("original: S_max=2000, upper BC = 0", 2000, "zero",  'rgb(255, 75, 75)'),
    ("fixed: S_max=150, upper BC = S−Ke^(−r(T−t))", 150, "linear", 'rgb(0, 191, 255)'),
]
fig = make_subplots(rows=1, cols=3, subplot_titles=[c[0] for c in cases], horizontal_spacing=0.06)
exact_today = bs_call_exact(100.0)
print(f"{'case':<48} {'V(100)':>8} {'error':>8} {'max |error| on 70–130':>22} {'grid points in 70–150':>22}")
for col, (label, S_max, bc, color) in enumerate(cases, start=1):
    S, t, V, _ = bs_explicit_fd(S_max, M=100, N=1000, upper_bc=bc)
    V0 = V[:, 0]
    near = (S >= 70) & (S <= 130)
    v100 = np.interp(100, S, V0)
    print(f"{label:<48} {v100:>8.4f} {v100 - exact_today:>+8.4f} {np.abs(V0[near] - bs_call_exact(S[near])).max():>22.4f} "
          f"{((S >= 70) & (S <= 150)).sum():>22}")
    S_plot = np.linspace(0, 150, 600)
    fig.add_trace(go.Scatter(x=S_plot, y=bs_call_exact(S_plot), mode='lines', line=dict(color='rgb(57, 255, 20)', width=4),
                             opacity=0.5, name='analytical', showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=S, y=V0, mode='lines+markers', marker=dict(size=4), line=dict(color=color, width=2),
                             name='finite differences', showlegend=(col == 1)), row=1, col=col)
    fig.update_xaxes(range=[0, 150] if S_max <= 150 else [0, 200], title_text='Stock Price',
                     showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
    fig.update_yaxes(range=[-2, 60], showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.update_yaxes(title_text='Option Value today, V(S, 0)', row=1, col=1)
fig.update_layout(width=1500, height=500, title_text='Black-Scholes Option Pricing: Finite Differences vs Analytical',
                  title_x=0.5, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(x=0.01, y=0.98, bgcolor='rgba(0,0,0,0)'))
fig.show()
print(f"\nClosed form V(100) = {exact_today:.4f}")

###### ______________________________________________________________________________________________________________________________________

##### 🧮 격자 계산을 눈으로 보기: 한 열씩, 뒤에서 앞으로

아래 애니메이션은 **일부러 아주 거친 격자**($S_{max}=200$, $M=20$ 이라 $\Delta S=10$, $N=20$ 이라 $\Delta t=0.05$)를 씁니다. 칸 441개를 하나하나 볼 수 있습니다.

**왼쪽 그림 = 격자 그 자체**
- 가로축은 시간 $t$ (오른쪽 끝이 만기 $T$), 세로축은 주가 $S$. 점 하나가 표의 칸 하나 $V_i^j$ 입니다.
- 모양: ◆ 만기열(payoff, 처음부터 앎), ■ 경계행($S=0$, $S=S_{max}$, 공식으로 앎), ● 내부(계산해야 함)
- 빈 점은 아직 모르는 칸이고, 채워진 점은 계산된 칸입니다. **색 = 옵션 가치**.
- 매 프레임 **한 열**이 채워집니다. $S=K=100$ 인 칸에 대해 흰 선 세 개가 "이 칸은 오른쪽 열의 이 세 칸으로 계산됐다"를 보여주고, 위쪽 제목에 실제 숫자 계산식이 나옵니다.

**오른쪽 그림 = 방금 채운 열 하나를 세로로 세운 것**, 즉 그 시점의 가격곡선 $V(S, t_j)$ 입니다.
- 점선: 만기 payoff. 파란 점: 유한차분. 초록 선: 같은 시점의 BS 해석해.
- 뒤로 갈수록 payoff의 **꺾인 점이 열처럼 퍼져 매끈해지는** 과정이 보입니다. 만기까지 시간이 남아 있으면 OTM 콜도 가치가 있다는 뜻입니다.

▶️ Play를 누르거나 슬라이더를 끌어 보세요.

In [ ]:
# ============================================================
# 🧮 Watch the grid being filled, column by column (backward in time)
# ============================================================
S_max_g, M_g, N_g = 200, 20, 20                    # deliberately coarse: every node is visible
S_g, t_g, V_g, (a_g, b_g, c_g) = bs_explicit_fd(S_max_g, M_g, N_g)
dS_g, dt_g = S_max_g / M_g, T / N_g
i_star = int(round(K / dS_g))                      # the row where S = K = 100
print(f"ΔS = {dS_g}, Δt = {dt_g}, stability limit Δt ≤ {1 / (sigma**2 * M_g**2 + r):.4f}  →  OK")
print(f"Weights for the row S = {S_g[i_star]:.0f} (i = {i_star}):  a = {a_g[i_star]:.4f},  b = {b_g[i_star]:.4f},  c = {c_g[i_star]:.4f}")

TT, SS = np.meshgrid(t_g, S_g)                     # node coordinates, shape (M+1, N+1)
kind = np.full(V_g.shape, "circle", dtype=object)  # interior
kind[:, -1] = "diamond"                            # terminal column (payoff)
kind[0, :] = "square"                              # S = 0 boundary
kind[-1, :] = "square"                             # S = S_max boundary

fig = make_subplots(rows=1, cols=2, column_widths=[0.58, 0.42], horizontal_spacing=0.12,
                    subplot_titles=("The grid: each dot is one unknown V(S_i, t_j)",
                                    "The column just filled = price curve V(S, t_j)"))
base_titles = list(fig.layout.annotations)
cmax = S_max_g - K

# 0: every node as an open outline ("not computed yet")
fig.add_trace(go.Scatter(x=TT.ravel(), y=SS.ravel(), mode='markers', hoverinfo='skip', name='not computed yet',
                         marker=dict(symbol=[k + '-open' for k in kind.ravel()], size=9, color='rgba(200,200,200,0.35)')),
              row=1, col=1)

def filled_nodes(j):
    # nodes in columns j..N are known once column j has been computed
    return dict(x=TT[:, j:].ravel(), y=SS[:, j:].ravel(),
                marker=dict(color=V_g[:, j:].ravel(), symbol=list(kind[:, j:].ravel()), size=11,
                            colorscale='Viridis', cmin=0, cmax=cmax, line=dict(width=0),
                            colorbar=dict(title='V', x=0.535, len=0.85, thickness=12)),
                text=[f"S = {s:.0f}, t = {tt:.2f}<br>V = {v:.3f}" for s, tt, v in
                      zip(SS[:, j:].ravel(), TT[:, j:].ravel(), V_g[:, j:].ravel())])

def stencil(j):
    # three arrows from column j+1 into the node (i_star, j)
    if j >= N_g:
        return dict(x=[], y=[]), dict(x=[], y=[])
    xs_, ys_ = [], []
    for di in (-1, 0, 1):
        xs_ += [t_g[j + 1], t_g[j], None]
        ys_ += [S_g[i_star + di], S_g[i_star], None]
    pts = dict(x=[t_g[j + 1]] * 3 + [t_g[j]], y=[S_g[i_star - 1], S_g[i_star], S_g[i_star + 1], S_g[i_star]])
    return dict(x=xs_, y=ys_), pts

def caption(j):
    if j == N_g:
        return (f"t = T = {T:.2f}: the right-hand column is the payoff max(S − {K}, 0), known before any calculation.<br>"
                f"The rows S = 0 and S = {S_max_g} are known from the boundary conditions.")
    Vd, Vm, Vu = V_g[i_star - 1, j + 1], V_g[i_star, j + 1], V_g[i_star + 1, j + 1]
    return (f"t = {t_g[j]:.2f}:  V({S_g[i_star]:.0f}, {t_g[j]:.2f}) = a·V↓ + b·V→ + c·V↑ = "
            f"{a_g[i_star]:.4f}×{Vd:.3f} + {b_g[i_star]:.4f}×{Vm:.3f} + {c_g[i_star]:.4f}×{Vu:.3f} = <b>{V_g[i_star, j]:.4f}</b><br>"
            f"(every other interior node in this column is computed the same way from ITS three neighbours)")

def dyn_annotation(j):
    return go.layout.Annotation(text=caption(j), x=0.0, y=1.2, xref='paper', yref='paper', xanchor='left',
                                showarrow=False, align='left', font=dict(size=13, color='white'))

# 1: filled nodes, 2: stencil lines, 3: stencil endpoints
fn = filled_nodes(N_g)
fig.add_trace(go.Scatter(mode='markers', name='computed (colour = V)', hovertemplate='%{text}<extra></extra>', **fn), row=1, col=1)
lines0, pts0 = stencil(N_g)
fig.add_trace(go.Scatter(mode='lines', line=dict(color='white', width=3), name='V↓, V→, V↑ → new node', **lines0), row=1, col=1)
fig.add_trace(go.Scatter(mode='markers', marker=dict(size=18, symbol='circle-open', color='white', line=dict(width=3)),
                         hoverinfo='skip', showlegend=False, **pts0), row=1, col=1)

# right panel: 4 payoff (static), 5 analytical at t_j, 6 finite differences column j
S_fine = np.linspace(0, S_max_g, 400)
fig.add_trace(go.Scatter(x=S_fine, y=np.maximum(S_fine - K, 0), mode='lines', line=dict(color='rgb(255, 165, 0)', dash='dash', width=2),
                         name='payoff at T'), row=1, col=2)
fig.add_trace(go.Scatter(x=S_fine, y=bs_call_exact(S_fine, tau=0.0), mode='lines', line=dict(color='rgb(57, 255, 20)', width=4),
                         opacity=0.5, name='analytical at t_j'), row=1, col=2)
fig.add_trace(go.Scatter(x=S_g, y=V_g[:, N_g], mode='markers+lines', marker=dict(size=8, color='rgb(0, 191, 255)'),
                         line=dict(color='rgb(0, 191, 255)', width=1), name='finite differences, column j'), row=1, col=2)

frames = []
for k in range(N_g + 1):
    j = N_g - k                                    # column filled in this frame
    lines, pts = stencil(j)
    frames.append(go.Frame(
        name=str(k), traces=[1, 2, 3, 5, 6],
        data=[go.Scatter(**filled_nodes(j)), go.Scatter(**lines), go.Scatter(**pts),
              go.Scatter(x=S_fine, y=bs_call_exact(S_fine, tau=T - t_g[j])), go.Scatter(x=S_g, y=V_g[:, j])],
        layout=go.Layout(annotations=base_titles + [dyn_annotation(j)])))
fig.frames = frames

fig.update_layout(
    annotations=base_titles + [dyn_annotation(N_g)],
    width=1350, height=700, margin=dict(t=150, b=130),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    legend=dict(orientation='h', x=0.0, y=-0.13, bgcolor='rgba(0,0,0,0)'),
    updatemenus=[{'type': 'buttons', 'showactive': False, 'x': 0.0, 'y': -0.22, 'xanchor': 'left', 'direction': 'left',
                  'buttons': [{'label': '▶️ Play', 'method': 'animate',
                               'args': [None, {'frame': {'duration': 900, 'redraw': True}, 'fromcurrent': True}]},
                              {'label': '⏸ Pause', 'method': 'animate',
                               'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]}]}],
    sliders=[{'x': 0.15, 'len': 0.85, 'y': -0.17, 'currentvalue': {'prefix': 'columns filled (from the right): '},
              'steps': [{'label': str(k), 'method': 'animate',
                         'args': [[str(k)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}]}
                        for k in range(N_g + 1)]}],
)
grid_style = dict(showgrid=True, gridcolor='rgba(128,128,128,0.2)', zeroline=False)
fig.update_xaxes(grid_style, title_text='time t  (← today ... maturity →)', range=[-0.05, T + 0.05], row=1, col=1)
fig.update_yaxes(grid_style, title_text='stock price S', range=[-8, S_max_g + 8], row=1, col=1)
fig.update_xaxes(grid_style, title_text='stock price S', range=[0, S_max_g], row=1, col=2)
fig.update_yaxes(grid_style, title_text='option value V', range=[-3, cmax + 10], row=1, col=2)
fig.show()

##### 🔍 마지막 한 칸을 손으로 계산해보기, 그리고 이 계산의 정체: 삼항트리

위 애니메이션의 마지막 프레임(오늘, $t=0$)에서 $S=100$ 칸이 어떻게 계산됐는지 숫자로 다시 따라가 봅니다.

그리고 가중치 $a, b, c$ 를 **확률**로 읽어보겠습니다.

$$a_i + b_i + c_i = 1 - r\Delta t \approx e^{-r\Delta t}$$

세 가중치의 합이 **할인계수**입니다. 그래서 $p_d = \frac{a_i}{1-r\Delta t},\ p_m = \frac{b_i}{1-r\Delta t},\ p_u = \frac{c_i}{1-r\Delta t}$ 로 놓으면

$$V_i^j = (1-r\Delta t)\,\big[\,p_d\,V_{i-1}^{j+1} + p_m\,V_i^{j+1} + p_u\,V_{i+1}^{j+1}\,\big] \approx e^{-r\Delta t}\; E\big[V^{j+1}\big]$$

즉 **"오늘 가치 = 다음 순간 가치의 기댓값을 한 스텝 할인한 것"** 입니다. 이 확률로 주가가 한 칸 아래/그대로/한 칸 위로 움직일 때

- 평균 이동: $(p_u - p_d)\Delta S \approx rS\,\Delta t$. drift가 **$r$**, 즉 위험중립입니다.
- 분산: $(p_u + p_d)\Delta S^2 \approx \sigma^2 S^2\,\Delta t$. 변동성이 **$\sigma$** 입니다.

**explicit 유한차분은 위험중립 GBM을 한 스텝씩 흉내내는 삼항트리**이고, 각 칸에서 할인된 기댓값을 거꾸로 계산하는 것입니다.
안정성 조건($b_i \ge 0$)은 "확률은 음수가 될 수 없다"는 조건과 같습니다. 몬테카를로는 같은 기댓값을 경로를 앞으로 뿌려서 계산하고, 유한차분은 격자 위에서 뒤로 계산합니다. 맨 끝 셀에서 두 방법의 값이 같은지 확인합니다.

(작은 예외: 맨 아래 $i=1$ 에서는 $\sigma^2 i < r$ 이라 $a_1$ 이 아주 약간 음수입니다. $S=0$ 경계 바로 옆이라 결과에 영향은 거의 없습니다.)

In [ ]:
j0 = 0
Vd, Vm, Vu = V_g[i_star - 1, j0 + 1], V_g[i_star, j0 + 1], V_g[i_star + 1, j0 + 1]
ai, bi, ci = a_g[i_star], b_g[i_star], c_g[i_star]
print(f"Node: S = {S_g[i_star]:.0f}, t = {t_g[j0]:.2f}  (i = {i_star}, j = {j0})")
print(f"  neighbours at t = {t_g[j0 + 1]:.2f}:  V↓ (S={S_g[i_star - 1]:.0f}) = {Vd:.4f}   V→ (S={S_g[i_star]:.0f}) = {Vm:.4f}   V↑ (S={S_g[i_star + 1]:.0f}) = {Vu:.4f}")
print(f"  a·V↓ + b·V→ + c·V↑ = {ai:.5f}×{Vd:.4f} + {bi:.5f}×{Vm:.4f} + {ci:.5f}×{Vu:.4f} = {ai * Vd + bi * Vm + ci * Vu:.4f}")
print(f"  value stored in the grid                 = {V_g[i_star, j0]:.4f}")
print(f"  closed form at S = 100                   = {bs_call_exact(100.0):.4f}   (the coarse ΔS = 10 grid is off by {V_g[i_star, j0] - bs_call_exact(100.0):+.4f})\n")

disc = ai + bi + ci
p_d, p_m, p_u = ai / disc, bi / disc, ci / disc
S_i = S_g[i_star]
mean_move = (p_u - p_d) * dS_g
var_move = (p_u + p_d) * dS_g**2 - mean_move**2
tree = pd.DataFrame({
    "trinomial reading of (a, b, c)": [disc, p_d, p_m, p_u, mean_move, var_move],
    "target (risk-neutral GBM)": [np.exp(-r * dt_g), np.nan, np.nan, np.nan, r * S_i * dt_g, sigma**2 * S_i**2 * dt_g],
}, index=["a + b + c  vs  e^(−rΔt)", "p_down", "p_middle", "p_up",
          "mean one-step move of S  vs  rSΔt", "variance of the move  vs  σ²S²Δt"])
display(tree.round(5))

# Stability = all "probabilities" non-negative
print(f"Smallest middle weight b_i on this grid: {b_g[1:-1].min():.4f} (at the top row, S = {S_g[-2]:.0f}); must be ≥ 0")
print(f"→ Δt must satisfy Δt ≤ 1/(σ²M² + r) = {1 / (sigma**2 * M_g**2 + r):.4f}; here Δt = {dt_g}")

##### 🗻 격자 전체를 한눈에: $V(S, t)$ 곡면

이번에는 촘촘한 격자($\Delta S = 2$, $\Delta t = 0.001$)의 표 전체를 곡면으로 그립니다. 오른쪽 끝의 주황 선이 만기 payoff(출발점)이고, 계산은 이 선에서 시작해 초록 선(오늘의 가격곡선, 도착점)으로 진행됩니다. 만기에 가까울수록 $S=K$ 근처가 뾰족하고, 만기에서 멀어질수록 매끈해집니다.

In [ ]:
S_f, t_f, V_f, _ = bs_explicit_fd(S_max=300, M=150, N=1000)
keep_S = S_f <= 200
step_t = 20
fig = go.Figure()
fig.add_trace(go.Surface(x=t_f[::step_t], y=S_f[keep_S], z=V_f[keep_S][:, ::step_t], colorscale='Viridis', opacity=0.85,
                         showscale=True, colorbar=dict(title='V')))
fig.add_trace(go.Scatter3d(x=np.full(keep_S.sum(), T), y=S_f[keep_S], z=V_f[keep_S, -1], mode='lines',
                           line=dict(color='rgb(255, 165, 0)', width=8), name='t = T: payoff (start here)'))
fig.add_trace(go.Scatter3d(x=np.zeros(keep_S.sum()), y=S_f[keep_S], z=V_f[keep_S, 0], mode='lines',
                           line=dict(color='rgb(57, 255, 20)', width=8), name='t = 0: today\'s prices (end here)'))
axis = dict(gridcolor='darkgray', showgrid=True, color='darkgray', backgroundcolor='rgb(30, 30, 35)')
fig.update_layout(title='V(S, t) on the whole grid, solved from the payoff (back) to today (front)', width=1000, height=700,
                  paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'), legend=dict(x=0.0, y=0.95),
                  scene=dict(xaxis_title='t - Time', yaxis_title='S - Stock Price', zaxis_title='V - Option Value',
                             xaxis=axis, yaxis=axis, zaxis=axis, bgcolor='rgba(0,0,0,0)',
                             camera=dict(eye=dict(x=-1.6, y=-1.6, z=0.9))))
fig.show()

###### ______________________________________________________________________________________________________________________________________

##### 💥 안정성 조건을 어기면?

$S_{max}=200$, $M=100$ ($\Delta S = 2$) 이면 조건은 $\Delta t \le 1/(0.04\cdot 100^2 + 0.05) \approx 0.0025$, 즉 $N \ge 400$ 입니다.
- $N = 400$: 조건을 만족하므로 안정적입니다.
- $N = 300$: 위쪽 행에서 $b_i < 0$ 이 됩니다(음수 확률). 톱니 모양 오차가 매 스텝 증폭돼 결과가 완전히 망가집니다.

가중치가 음수가 되는 건 안정성의 **충분조건을 어기는 것**이라, 아주 살짝 어기면 운 좋게 버틸 때도 있습니다. 하지만 보장이 없으니 항상 조건을 지키는 게 원칙입니다. (이 제약이 없는 **implicit / Crank–Nicolson** 방법은 각 시점마다 연립방정식을 푸는 대가로 큰 $\Delta t$ 를 쓸 수 있습니다.)

In [ ]:
fig = make_subplots(rows=1, cols=2, subplot_titles=("Today's price curve V(S, 0)", "max |V| in each column while marching back (log)"))
S_plot = np.linspace(0, 200, 400)
fig.add_trace(go.Scatter(x=S_plot, y=bs_call_exact(S_plot), mode='lines', line=dict(color='rgb(57, 255, 20)', width=5),
                         opacity=0.5, name='analytical'), row=1, col=1)
for N_try, color in [(400, 'rgb(0, 191, 255)'), (300, 'rgb(255, 75, 75)')]:
    S_s, t_s, V_s, (a_s, b_s, c_s) = bs_explicit_fd(S_max=200, M=100, N=N_try)
    dt_s = T / N_try
    print(f"N = {N_try}: Δt = {dt_s:.5f}  (limit {1 / (sigma**2 * 100**2 + r):.5f}),  min b_i = {b_s[1:-1].min():+.4f},  "
          f"V(100, 0) = {V_s[50, 0]:.6g}   (exact {bs_call_exact(100.0):.4f})")
    fig.add_trace(go.Scatter(x=S_s, y=V_s[:, 0], mode='lines', line=dict(color=color, width=2),
                             name=f'N = {N_try}, Δt = {dt_s:.4f}'), row=1, col=1)
    fig.add_trace(go.Scatter(x=t_s[::-1], y=np.nanmax(np.abs(V_s), axis=0)[::-1], mode='lines', line=dict(color=color, width=2),
                             showlegend=False), row=1, col=2)
fig.update_yaxes(range=[-20, 120], title_text='V', row=1, col=1)
fig.update_xaxes(title_text='S', row=1, col=1)
fig.update_yaxes(type='log', exponentformat='power', title_text='max |V|', row=1, col=2)
fig.update_xaxes(title_text='t (computed from right to left)', autorange='reversed', row=1, col=2)
fig.update_layout(width=1300, height=480, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(x=0.01, y=0.98, bgcolor='rgba(0,0,0,0)'))
for col in (1, 2):
    fig.update_xaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.show()

##### 📉 격자를 촘촘하게 하면 얼마나 정확해지나

$\Delta S$ 를 반으로 줄일 때마다 안정성 때문에 $\Delta t$ 는 4분의 1로 줄입니다($\Delta t \propto \Delta S^2$). 공간은 중앙차분이라 오차가 $O(\Delta S^2)$, 시간은 $O(\Delta t) = O(\Delta S^2)$ 이므로 **$\Delta S$ 를 반으로 줄이면 오차는 약 4분의 1**이 되어야 합니다. 대신 계산량은 약 8배($M$ 2배 × $N$ 4배)로 늘어납니다.

In [ ]:
import time
rows = []
prev_err = None
for M_c in (20, 40, 80, 160, 320):
    N_c = int(np.ceil(1.25 * (sigma**2 * M_c**2 + r) * T))   # 20% safety margin under the stability limit
    t0 = time.perf_counter()
    S_c, _, V_c, _ = bs_explicit_fd(S_max=200, M=M_c, N=N_c)
    elapsed = time.perf_counter() - t0
    i100 = int(round(100 / (200 / M_c)))                     # S = 100 is a grid node for all these M
    err = V_c[i100, 0] - bs_call_exact(100.0)
    rows.append((M_c, N_c, 200 / M_c, T / N_c, (M_c + 1) * (N_c + 1), V_c[i100, 0], err,
                 np.nan if prev_err is None else prev_err / err, elapsed * 1000))
    prev_err = err
conv = pd.DataFrame(rows, columns=["M", "N", "ΔS", "Δt", "grid cells", "V(100, 0)", "error", "error ratio vs previous", "time (ms)"])
print(f"Closed form: {bs_call_exact(100.0):.6f}")
display(conv.style.format({"ΔS": "{:.3f}", "Δt": "{:.6f}", "grid cells": "{:,}", "V(100, 0)": "{:.6f}",
                           "error": "{:+.6f}", "error ratio vs previous": "{:.2f}", "time (ms)": "{:.1f}"}))

##### 🎲 같은 가격, 세 가지 방법: 해석해 vs 유한차분 vs 몬테카를로

**Feynman–Kac 정리**: BS PDE의 해는 다음 기댓값과 같습니다.

$$V(S, t) = e^{-r(T-t)}\;E^{\mathbb{Q}}\big[\max(S_T - K, 0)\ \big|\ S_t = S\big], \qquad dS = rS\,dt + \sigma S\,dW^{\mathbb{Q}}$$

| 방법 | 기댓값을 계산하는 방식 |
|---|---|
| 해석해 | 로그정규분포로 적분을 끝까지 풀어낸 공식 |
| 유한차분 (PDE) | 격자 위에서 **뒤로**, 칸마다 삼항트리 기댓값을 한 스텝씩 |
| 몬테카를로 | 위험중립 GBM 경로를 **앞으로** 뿌려 만기 payoff를 평균 |

세 값이 (각자의 오차 범위 안에서) 같아야 합니다.

In [ ]:
rng = np.random.default_rng(0)
n_mc = 2_000_000
S0 = 100.0
Z = rng.standard_normal(n_mc)
S_T = S0 * np.exp((r - 0.5 * sigma**2) * T + sigma * np.sqrt(T) * Z)   # exact GBM terminal draw under Q
disc_payoff = np.exp(-r * T) * np.maximum(S_T - K, 0)
mc_price, mc_se = disc_payoff.mean(), disc_payoff.std(ddof=1) / np.sqrt(n_mc)

S_c, _, V_c, _ = bs_explicit_fd(S_max=200, M=320, N=int(np.ceil(1.25 * (sigma**2 * 320**2 + r) * T)))
fd_price = V_c[160, 0]
exact = bs_call_exact(S0)

print(f"Closed form               : {exact:.4f}")
print(f"Finite differences (PDE)  : {fd_price:.4f}   (ΔS = {200 / 320:.3f}; error {fd_price - exact:+.4f})")
print(f"Monte Carlo ({n_mc:,} paths): {mc_price:.4f} ± {1.96 * mc_se:.4f} (95%)   (error {mc_price - exact:+.4f})")

###### ______________________________________________________________________________________________________________________________________

##### 💡 그래서 왜 굳이 유한차분을 쓰나? 공식으로는 안 되는 두 가지

BS 유럽형 콜은 공식이 있으니 유한차분이 필요 없어 보입니다. 하지만 실무에서 유한차분을 쓰는 이유가 여기서 드러납니다.

**① 그릭스가 공짜로 나온다.** 트레이더에게는 가격만 필요한 게 아닙니다. 헤지를 하려면 **델타(Δ, 주가 민감도), 감마(Γ, 델타의 변화), 세타(Θ, 시간 경과에 따른 가치 변화)**가 필요합니다. 격자에는 모든 주가·시점의 가격이 이미 들어 있으니, 이웃 칸끼리 빼기만 하면 됩니다.

$$\Delta \approx \frac{V_{i+1}^0 - V_{i-1}^0}{2\Delta S},\qquad \Gamma \approx \frac{V_{i+1}^0 - 2V_i^0 + V_{i-1}^0}{\Delta S^2},\qquad \Theta \approx \frac{V_i^1 - V_i^0}{\Delta t}$$

몬테카를로로 같은 일을 하려면 주가를 조금씩 바꿔가며 시뮬레이션을 여러 번 다시 돌려야 하고, 결과에 노이즈도 섞입니다.

**② 공식이 없는 상품도 한 줄로 풀린다: 아메리칸 풋.** 아메리칸 옵션은 만기 전 아무 때나 행사할 수 있습니다. 매 순간 "**지금 행사할까**(행사가치 $K - S$), **계속 들고 있을까**(계속보유가치)"를 비교해야 해서 **닫힌 공식이 없습니다.** 유한차분에서는 칸을 계산할 때마다 한 줄만 추가하면 됩니다.

$$V_i^j = \max\Big(\underbrace{a_iV_{i-1}^{j+1} + b_iV_i^{j+1} + c_iV_{i+1}^{j+1}}_{\text{계속 보유할 때의 가치}},\ \underbrace{K - S_i}_{\text{지금 행사할 때의 가치}}\Big)$$

뒤에서부터 계산하기 때문에 "다음 시점의 가치"를 이미 알고 있고, 그래서 이 비교가 가능합니다. 경로를 앞으로 뿌리는 몬테카를로는 미래를 모르는 상태에서 행사 여부를 정해야 해서 회귀 같은 추가 기법(Longstaff–Schwartz)이 필요합니다.

(참고: 배당이 없으면 아메리칸 **콜**은 조기행사할 이유가 없어 유럽형과 가격이 같습니다. 그래서 차이가 나는 **풋**으로 보여줍니다.)

In [ ]:
# ============================================================
# ① Greeks read straight off the grid
# ============================================================
M_gk = 320
N_gk = int(np.ceil(1.25 * (sigma**2 * M_gk**2 + r) * T))
S_gk, t_gk, V_gk, _ = bs_explicit_fd(S_max=200, M=M_gk, N=N_gk)
dS_gk, dt_gk = 200 / M_gk, T / N_gk

S_in = S_gk[1:-1]
delta_fd = (V_gk[2:, 0] - V_gk[:-2, 0]) / (2 * dS_gk)                  # neighbours above/below
gamma_fd = (V_gk[2:, 0] - 2 * V_gk[1:-1, 0] + V_gk[:-2, 0]) / dS_gk**2
theta_fd = (V_gk[1:-1, 1] - V_gk[1:-1, 0]) / dt_gk                      # next time column minus today

d1 = (np.log(S_in / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
d2 = d1 - sigma * np.sqrt(T)
delta_ex = norm.cdf(d1)
gamma_ex = norm.pdf(d1) / (S_in * sigma * np.sqrt(T))
theta_ex = -S_in * norm.pdf(d1) * sigma / (2 * np.sqrt(T)) - r * K * np.exp(-r * T) * norm.cdf(d2)

k100 = int(np.argmin(np.abs(S_in - 100)))
greeks = pd.DataFrame({"from the grid": [V_gk[k100 + 1, 0], delta_fd[k100], gamma_fd[k100], theta_fd[k100]],
                       "closed form": [bs_call_exact(100.0), delta_ex[k100], gamma_ex[k100], theta_ex[k100]]},
                      index=["price V", "Delta ∂V/∂S", "Gamma ∂²V/∂S²", "Theta ∂V/∂t (per year)"])
print("Call at S = 100: one grid solve gives the price AND the Greeks")
display(greeks.round(5))

fig = make_subplots(rows=1, cols=3, subplot_titles=("Delta", "Gamma", "Theta (per year)"), horizontal_spacing=0.07)
show = (S_in >= 50) & (S_in <= 150)
for col, (fd_vals, ex_vals) in enumerate([(delta_fd, delta_ex), (gamma_fd, gamma_ex), (theta_fd, theta_ex)], start=1):
    fig.add_trace(go.Scatter(x=S_in[show], y=ex_vals[show], mode='lines', line=dict(color='rgb(57, 255, 20)', width=5),
                             opacity=0.5, name='closed form', showlegend=(col == 1)), row=1, col=col)
    fig.add_trace(go.Scatter(x=S_in[show][::8], y=fd_vals[show][::8], mode='markers', marker=dict(color='rgb(0, 191, 255)', size=6),
                             name='from the grid (neighbour differences)', showlegend=(col == 1)), row=1, col=col)
    fig.update_xaxes(title_text='S', showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.update_layout(width=1400, height=430, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(x=0.0, y=-0.2, orientation='h', bgcolor='rgba(0,0,0,0)'))
fig.show()

# ============================================================
# ② American put: the same grid plus ONE line (early exercise)
# ============================================================
def put_explicit_fd(S_max, M, N, american, K=K, T=T, r=r, sigma=sigma):
    dt = T / N
    S = np.linspace(0, S_max, M + 1)
    t = np.linspace(0, T, N + 1)
    i = np.arange(M + 1)
    a = 0.5 * dt * (sigma**2 * i**2 - r * i)
    b = 1 - dt * (sigma**2 * i**2 + r)
    c = 0.5 * dt * (sigma**2 * i**2 + r * i)
    exercise_value = np.maximum(K - S, 0)
    V = exercise_value.copy()                                   # payoff at maturity
    boundary = np.full(N + 1, np.nan)                           # critical price S*(t): exercise below it
    boundary[N] = K
    for j in range(N - 1, -1, -1):
        hold = np.empty_like(V)
        hold[1:M] = a[1:M] * V[:M - 1] + b[1:M] * V[1:M] + c[1:M] * V[2:]   # value of waiting one more step
        hold[M] = 0.0                                                     # deep out-of-the-money put
        hold[0] = K if american else K * np.exp(-r * (T - t[j]))          # S = 0
        if american:
            V = np.maximum(hold, exercise_value)                          # ← the one extra line
            ex = (exercise_value > 0) & (hold <= exercise_value)
            boundary[j] = S[ex].max() if ex.any() else np.nan
        else:
            V = hold
    return S, V, t, boundary

S_p, V_eu, t_p, _ = put_explicit_fd(200, M_gk, N_gk, american=False)
_, V_am, _, S_star = put_explicit_fd(200, M_gk, N_gk, american=True)
i100 = int(round(100 / dS_gk))
eu_exact = bs_call_exact(100.0) - 100 + K * np.exp(-r * T)          # put-call parity
print(f"European put: grid {V_eu[i100]:.4f}   closed form {eu_exact:.4f}")
print(f"American put: grid {V_am[i100]:.4f}   (no closed form exists)")
print(f"Early-exercise premium = {V_am[i100] - V_eu[i100]:.4f}")
print(f"Today's critical price S*(0) = {S_star[0]:.2f}: below it, exercising immediately beats waiting")

fig = make_subplots(rows=1, cols=2, subplot_titles=("Put value today V(S, 0)", "Early-exercise boundary S*(t)"))
view = (S_p >= 50) & (S_p <= 150)
fig.add_trace(go.Scatter(x=S_p[view], y=np.maximum(K - S_p[view], 0), mode='lines', line=dict(color='rgb(255, 165, 0)', dash='dash'),
                         name='exercise value max(K − S, 0)'), row=1, col=1)
fig.add_trace(go.Scatter(x=S_p[view], y=V_eu[view], mode='lines', line=dict(color='rgb(0, 191, 255)', width=2),
                         name='European put'), row=1, col=1)
fig.add_trace(go.Scatter(x=S_p[view], y=V_am[view], mode='lines', line=dict(color='rgb(57, 255, 20)', width=3),
                         name='American put'), row=1, col=1)
fig.add_trace(go.Scatter(x=t_p, y=S_star, mode='lines', line=dict(color='rgb(255, 75, 75)', width=3), fill='tozeroy',
                         fillcolor='rgba(255, 75, 75, 0.15)', name='exercise region (S below S*)'), row=1, col=2)
fig.update_xaxes(title_text='S', row=1, col=1)
fig.update_yaxes(title_text='value', row=1, col=1)
fig.update_xaxes(title_text='t', row=1, col=2)
fig.update_yaxes(title_text='S*', range=[60, 105], row=1, col=2)
fig.update_layout(width=1300, height=480, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(x=0.2, y=0.98, bgcolor='rgba(0,0,0,0)'))
for col in (1, 2):
    fig.update_xaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.show()

**결과 읽는 법**
- **그릭스 표와 그림:** 격자 한 번 계산으로 가격과 델타·감마·세타가 공식과 거의 같게 나옵니다. 따로 계산한 게 아니라 이미 있는 격자 값을 이웃끼리 뺐을 뿐입니다.
- **왼쪽 풋 그림:** 아메리칸 풋(초록)은 항상 유럽형(파랑)보다 비싸거나 같습니다. 행사할 권리가 더 많기 때문입니다. 주가가 낮은 구간에서는 아메리칸 풋 가치가 행사가치(주황 점선)와 **딱 붙어** 있습니다. 거기서는 기다리는 것보다 지금 행사하는 게 낫다는 뜻입니다.
- **오른쪽 그림 (조기행사 경계):** 빨간 선 아래로 주가가 내려가면 즉시 행사하는 게 최적입니다. 만기가 다가올수록 경계가 행사가 $K$ 쪽으로 올라갑니다. 이 경계는 공식으로 구할 수 없고, 격자를 뒤에서부터 계산했기 때문에 얻을 수 있었습니다. (선이 계단 모양인 건 경계가 격자점 위에서만 정해지기 때문입니다. 간격 $\Delta S = 0.625$ 를 줄이면 매끈해집니다.)

###### ______________________________________________________________________________________________________________________________________

#### 5.) 🇰🇷 실전 예제: 기초자산 1개짜리 스텝다운 ELS를 격자로 평가하기

**🎯 이 섹션의 목적:** 지금까지 배운 것(뒤로 가는 격자, 경계조건, 특정 시점에 격자 값 덮어쓰기)을 한국에서 가장 많이 팔리는 구조화상품인 **스텝다운 ELS**에 그대로 적용합니다.

**왜 ELS에는 공식이 없나?**
1. **조기상환:** 6개월마다 "주가가 기준 이상이면 원금+쿠폰을 주고 끝"이라는 조건이 붙어 있어서, 상품이 언제 끝날지가 주가 경로에 따라 달라집니다.
2. **낙인(Knock-In):** 기간 중 **한 번이라도** 주가가 50% 아래로 내려간 적이 있는지에 따라 만기 상환금이 달라집니다. 즉 **경로에 의존**합니다.

앞의 아메리칸 풋처럼 "특정 시점에 조건을 적용"하는 상품이라 유한차분법이 잘 맞습니다. 실무에서도 기초자산 1~2개짜리 ELS는 유한차분법으로, 3개 이상은 몬테카를로로 평가하는 경우가 많습니다(기초자산 2개면 격자가 2차원이 되어 ADI 같은 기법을 씁니다).

##### 📄 예제 상품 (가상의 조건)

| 항목 | 조건 |
|---|---|
| 기초자산 | 1개 (가상의 주가 또는 지수, 최초기준가격 $S_0 = 100$) |
| 만기 | 3년 |
| 조기상환 평가 | 6개월마다 (총 6회, 마지막이 만기) |
| 상환 기준 (최초기준가격 대비) | 90% - 90% - 85% - 85% - 80% - 75%, 점점 내려가서 "스텝다운" |
| 쿠폰 | 연 6%. 6개월마다 3%씩 쌓여서 조기상환 시 103, 106, …, 만기 118 |
| 낙인 배리어 | 50%, 매 거래일 종가로 관찰 |
| 가격 모델 | Black-Scholes, $r = 3\%$, $\sigma = 25\%$, 배당 없음 |

**상환 규칙 (액면 100 기준)**

| 시점 | 조건 | 받는 금액 |
|---|---|---|
| $k$번째 평가일 $t_k$ ($k=1,\dots,5$) | $S_{t_k} \ge$ 그 회차 상환 기준 | $100\,(1 + 0.06\,t_k)$ 받고 **종료** |
| 만기 $T=3$ | $S_T \ge 75$ | $118$ |
| 만기 | $S_T < 75$, 기간 중 낙인 **없음** | $100$ (원금) |
| 만기 | $S_T < 75$, 기간 중 낙인 **있음** | $100\,S_T/S_0$ (**손실**. 예: 주가가 40%면 40) |

아래 그림은 상품 구조(계단 모양의 상환 기준, 낙인선)와 네 가지 대표 시나리오의 주가 경로입니다.

In [ ]:
# ============================================================
# 🇰🇷 Step-down ELS: contract terms, a path-by-path payoff engine, and the product picture
# ============================================================
els = dict(
    S0=100.0, T=3.0, r=0.03, sigma=0.25, days_per_year=252,
    obs_times=[0.5, 1.0, 1.5, 2.0, 2.5, 3.0],   # early-redemption dates (the last one is maturity)
    barriers=[90, 90, 85, 85, 80, 75],          # redemption levels, % of S0
    coupon=0.06,                                # per year: redeeming at t_k pays 100 * (1 + coupon * t_k)
    ki=50.0,                                    # knock-in level, % of S0, checked at every daily close
)
SCENARIOS = [f"early redemption #{k + 1} (t = {t}y)" for k, t in enumerate(els["obs_times"][:-1])] + \
            ["maturity, S_T ≥ 75 → 118", "maturity, S_T < 75, no knock-in → 100", "maturity, S_T < 75, knocked in → loss"]

def simulate_daily_paths(p, n_paths, rng):
    # Risk-neutral GBM at daily closes, column 0 = today
    days = int(round(p["T"] * p["days_per_year"]))
    dt = 1 / p["days_per_year"]
    steps = (p["r"] - 0.5 * p["sigma"]**2) * dt + p["sigma"] * np.sqrt(dt) * rng.standard_normal((n_paths, days))
    return p["S0"] * np.exp(np.hstack([np.zeros((n_paths, 1)), np.cumsum(steps, axis=1)]))

def els_outcome(p, paths):
    # Apply the contract rules to each path: returns (amount paid, payment time, scenario code)
    n = paths.shape[0]
    amount, when, code_ = np.zeros(n), np.full(n, p["T"]), np.full(n, -1)
    alive = np.ones(n, dtype=bool)
    for k, (t_k, B_k) in enumerate(zip(p["obs_times"], p["barriers"])):
        d = int(round(t_k * p["days_per_year"]))
        hit = alive & (paths[:, d] >= B_k / 100 * p["S0"])
        amount[hit], when[hit], code_[hit] = 100 * (1 + p["coupon"] * t_k), t_k, k
        alive &= ~hit
    knocked_in = paths[:, 1:].min(axis=1) <= p["ki"] / 100 * p["S0"]
    par, loss = alive & ~knocked_in, alive & knocked_in          # still alive at maturity ⇒ S_T < 75
    amount[par], code_[par] = 100.0, len(p["obs_times"])
    amount[loss], code_[loss] = 100 * paths[loss, -1] / p["S0"], len(p["obs_times"]) + 1
    return amount, when, code_

# --- The product picture with one example path per scenario ---
rng = np.random.default_rng(3)
demo_paths = simulate_daily_paths(els, 4000, rng)
demo_amount, demo_when, demo_code = els_outcome(els, demo_paths)
days_axis = np.arange(demo_paths.shape[1]) / els["days_per_year"]

fig = go.Figure()
t_prev = 0.0
for k, (t_k, B_k) in enumerate(zip(els["obs_times"], els["barriers"])):
    fig.add_trace(go.Scatter(x=[t_prev, t_k], y=[B_k, B_k], mode='lines', line=dict(color='white', width=3),
                             showlegend=(k == 0), name='redemption level (steps down)'))
    fig.add_annotation(x=t_k, y=B_k, text=f"{B_k}% → {100 * (1 + els['coupon'] * t_k):.0f}", showarrow=False,
                       yshift=14, xanchor='right', font=dict(color='white'))
    t_prev = t_k
fig.add_hline(y=els["ki"], line=dict(color='rgb(255, 75, 75)', dash='dash', width=2),
              annotation_text="knock-in 50% (daily close)", annotation_position="bottom left", annotation_font_color='rgb(255, 75, 75)')
examples = [(0, 'rgb(57, 255, 20)'), (3, 'rgb(0, 191, 255)'), (6, 'rgb(255, 209, 102)'), (7, 'rgb(255, 75, 75)')]
for scen, color in examples:
    idx = np.flatnonzero(demo_code == scen)[0]
    end = int(round(demo_when[idx] * els["days_per_year"]))
    fig.add_trace(go.Scatter(x=days_axis[:end + 1], y=demo_paths[idx, :end + 1], mode='lines', line=dict(color=color, width=1.5),
                             name=f"{SCENARIOS[scen]}: pays {demo_amount[idx]:.1f}"))
    fig.add_trace(go.Scatter(x=[demo_when[idx]], y=[demo_paths[idx, end]], mode='markers', marker=dict(color=color, size=11),
                             showlegend=False))
fig.update_layout(title="Step-down ELS: redemption levels, knock-in line and four example paths", width=1200, height=560,
                  plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  xaxis_title='time (years)', yaxis_title='underlying (% of initial)',
                  legend=dict(x=0.01, y=0.99, bgcolor='rgba(0,0,0,0.4)'))
fig.update_xaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', dtick=0.5)
fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', range=[20, 180])
fig.show()

##### 🧮 격자에 올리는 방법: 핵심 아이디어 세 가지

**① 만기에서 출발해 뒤로 간다 (지금까지와 같음).** 평가일과 평가일 사이에는 아무 일도 일어나지 않으니, 이 구간에서 ELS 가치는 **그냥 BS PDE**를 따릅니다. 앞에서 만든 가중합 한 스텝 $V_i^j = a_iV_{i-1}^{j+1} + b_iV_i^{j+1} + c_iV_{i+1}^{j+1}$ 을 그대로 씁니다.

**② 특별한 날에는 격자 값을 "덮어쓴다".** 뒤로 가다가 조기상환 평가일 $t_k$ 에 도착하면, 그날 주가가 기준 이상인 칸($S_i \ge B_k$)에서는 상품이 끝나고 $100(1+c\,t_k)$ 를 받습니다. 그래서 그 칸들의 값을 이 금액으로 **바꿔 써넣습니다.** 기준 미만인 칸은 상품이 계속되니 계산된 값을 그대로 둡니다.
→ 앞의 아메리칸 풋에서 매 스텝 `max(계속 보유, 지금 행사)` 로 덮어쓴 것과 같은 원리입니다. 뒤에서부터 계산하기 때문에 "그 이후의 가치"를 이미 알고 있어서 가능합니다.

**③ 경로 의존(낙인)은 격자를 두 장 써서 해결한다.** 낙인 여부는 오늘 주가 $S$ 만으로는 알 수 없는 **과거의 정보**입니다. 그래서 상태를 둘로 나눠 각각 격자를 만듭니다.

| 격자 | 뜻 | 만기 값 |
|---|---|---|
| $V^{\text{KI}}(S,t)$ | 이미 낙인된 상태 | $S_T \ge 75$ 면 118, 아니면 $100\,S_T/S_0$ |
| $V^{\text{noKI}}(S,t)$ | 아직 낙인되지 않은 상태 | $S_T \ge 75$ 면 118, 아니면 100 |

두 격자를 **동시에** 뒤로 계산하다가, 낙인을 관찰하는 날(매 거래일 종가)마다 **"아직 낙인 안 됨" 격자에서 $S_i \le 50$ 인 칸을 "낙인됨" 격자 값으로 바꿔 씁니다.** 그날 주가가 50 이하라면 그 순간 낙인 상태로 넘어가기 때문입니다. 조기상환 평가일에는 두 격자 모두에 ②를 적용합니다.

오늘 주가 100은 낙인 전이므로 **오늘의 ELS 가격 = $V^{\text{noKI}}(100, 0)$** 입니다.

**경계조건**
- $S = 0$: 주가가 0이면 계속 0이라 낙인 손실로 0을 받습니다. 따라서 $V = 0$.
- $S = S_{max}$ (300): 다음 평가일에 확실히 조기상환되므로 $V = 100(1+c\,t_{next})\,e^{-r(t_{next}-t)}$.

**시간 격자:** 3년은 756 거래일입니다. 하루를 안정성 조건을 만족하는 만큼(여기서는 28개)의 작은 스텝으로 쪼개서, 낙인 관찰(하루 한 번)과 조기상환 평가일이 정확히 격자 시점에 오도록 맞춥니다.

In [ ]:
def price_stepdown_els(p, S_max=300.0, M=300, store_daily=False):
    S0, T, r_, sig = p["S0"], p["T"], p["r"], p["sigma"]
    S = np.linspace(0, S_max, M + 1)
    n_days = int(round(T * p["days_per_year"]))
    sub = int(np.ceil(1.25 * (sig**2 * M**2 + r_) * T / n_days))    # explicit steps per trading day (stability)
    N = n_days * sub
    dt = T / N
    i = np.arange(M + 1)
    a = 0.5 * dt * (sig**2 * i**2 - r_ * i)
    b = 1 - dt * (sig**2 * i**2 + r_)
    c = 0.5 * dt * (sig**2 * i**2 + r_ * i)

    redemption = [100 * (1 + p["coupon"] * t_k) for t_k in p["obs_times"]]
    obs_step = {int(round(t_k / dt)): k for k, t_k in enumerate(p["obs_times"])}
    below_ki = S <= p["ki"] / 100 * S0
    final_hit = S >= p["barriers"][-1] / 100 * S0

    # terminal values of the two grids
    V_ki = np.where(final_hit, redemption[-1], 100 * S / S0)
    V_no = np.where(final_hit, redemption[-1], 100.0)
    V_no[below_ki] = V_ki[below_ki]                                   # knocked in on the last day

    def one_step(V):                                                  # plain Black-Scholes step (weights a, b, c)
        W = np.empty_like(V)
        W[1:M] = a[1:M] * V[:M - 1] + b[1:M] * V[1:M] + c[1:M] * V[2:]
        return W

    snaps_t, snaps_no, snaps_ki = [T], [V_no.copy()], [V_ki.copy()]
    for j in range(N - 1, -1, -1):
        t = j * dt
        V_ki, V_no = one_step(V_ki), one_step(V_no)
        # boundaries
        k_next = next(k for k, t_k in enumerate(p["obs_times"]) if t_k > t + 1e-12)
        top = redemption[k_next] * np.exp(-r_ * (p["obs_times"][k_next] - t))
        V_ki[M] = V_no[M] = top
        V_ki[0] = V_no[0] = 0.0
        if j % sub == 0:                                              # a daily close: knock-in check
            V_no[below_ki] = V_ki[below_ki]
        if j in obs_step and obs_step[j] < len(redemption) - 1:       # an early-redemption date
            k = obs_step[j]
            hit = S >= p["barriers"][k] / 100 * S0
            V_ki[hit] = V_no[hit] = redemption[k]
        if store_daily and j % sub == 0:
            snaps_t.append(t); snaps_no.append(V_no.copy()); snaps_ki.append(V_ki.copy())
    out = dict(S=S, V_no=V_no, V_ki=V_ki, N=N, sub=sub, dS=S_max / M)
    if store_daily:
        out.update(t=np.array(snaps_t[::-1]), no=np.array(snaps_no[::-1]), ki=np.array(snaps_ki[::-1]))
    return out

t0 = time.perf_counter()
els_grid = price_stepdown_els(els, store_daily=True)
i0 = int(round(els["S0"] / els_grid["dS"]))
print(f"Grid: ΔS = {els_grid['dS']:.1f}, {els_grid['N']:,} time steps ({els_grid['sub']} per trading day), "
      f"2 grids × {len(els_grid['S']):,} nodes, {time.perf_counter() - t0:.2f} s")
print(f"ELS price today  V_noKI(S=100, t=0) = {els_grid['V_no'][i0]:.3f}  (per 100 of notional)")
print(f"If it had ALREADY knocked in:  V_KI(100, 0) = {els_grid['V_ki'][i0]:.3f}")

##### ✅ 몬테카를로로 검증하고, 시나리오별 확률 보기

같은 상품을 몬테카를로로도 평가해 격자 결과가 맞는지 확인합니다. 위험중립 GBM으로 매일 종가 경로를 만들고, 경로마다 상환 규칙을 그대로 적용한 뒤 할인해서 평균을 냅니다. 격자는 뒤로 계산하고 몬테카를로는 앞으로 경로를 뿌리지만, 같은 기댓값을 계산하므로 같은 가격이 나와야 합니다.

몬테카를로는 **어떤 시나리오로 끝날 확률**도 바로 보여줘서 상품을 이해하는 데 도움이 됩니다. (이 확률은 위험중립 확률입니다. 실제 확률은 주가의 실제 기대수익률에 따라 다릅니다.)

마지막 표는 격자 간격 $\Delta S$ 를 바꿔 가며 격자 가격이 몬테카를로 값에 가까워지는지 봅니다. 상환 기준에서 받는 금액이 **뚝 끊기기**(불연속) 때문에 오차가 매끈한 콜옵션처럼 $O(\Delta S^2)$ 로 빨리 줄지 않고 대략 $O(\Delta S)$ 로 줄어듭니다. 실무에서는 격자점을 배리어에 정확히 맞추거나(여기서도 맞춤), 불연속 지점의 payoff를 평활화해서 이 문제를 줄입니다.

몬테카를로 값에도 ± 오차가 있어서, 그 범위보다 작은 차이는 어느 쪽이 맞는지 구분할 수 없습니다. 그래서 100만 개 경로를 씁니다.

In [ ]:
rng = np.random.default_rng(2026)
n_mc, chunk = 1_000_000, 25_000
pv_all, amt_all, codes_all = [], [], []
for _ in range(n_mc // chunk):
    amount, when, code_ = els_outcome(els, simulate_daily_paths(els, chunk, rng))
    pv_all.append(amount * np.exp(-els["r"] * when))    # discount each payment from the date it is paid
    amt_all.append(amount)
    codes_all.append(code_)
pv_all, amt_all, codes_all = np.concatenate(pv_all), np.concatenate(amt_all), np.concatenate(codes_all)
mc_price, mc_se = pv_all.mean(), pv_all.std(ddof=1) / np.sqrt(n_mc)
print(f"Grid (FDM)  : {els_grid['V_no'][i0]:.3f}")
print(f"Monte Carlo : {mc_price:.3f} ± {1.96 * mc_se:.3f} (95%, {n_mc:,} paths)\n")

scen = pd.DataFrame({
    "risk-neutral probability": [(codes_all == k).mean() for k in range(len(SCENARIOS))],
    "average amount paid": [amt_all[codes_all == k].mean() for k in range(len(SCENARIOS))],
}, index=SCENARIOS)
display(scen.style.format({"risk-neutral probability": "{:.2%}", "average amount paid": "{:.1f}"}))

rows = []
for M_e in (150, 300, 600, 1200):
    t0 = time.perf_counter()
    g = price_stepdown_els(els, M=M_e)
    rows.append((M_e, g["dS"], g["N"], g["V_no"][int(round(100 / g["dS"]))], time.perf_counter() - t0))
conv_els = pd.DataFrame(rows, columns=["M", "ΔS", "time steps", "grid price", "time (s)"])
conv_els["grid − MC"] = conv_els["grid price"] - mc_price
print(f"Grid refinement (MC reference {mc_price:.3f} ± {1.96 * mc_se:.3f}):")
display(conv_els.style.format({"ΔS": "{:.2f}", "time steps": "{:,}", "grid price": "{:.3f}", "time (s)": "{:.2f}", "grid − MC": "{:+.3f}"}))

##### 🔍 격자 안을 들여다보기

**첫 그림 (히트맵):** 두 격자 $V^{\text{noKI}}(S,t)$, $V^{\text{KI}}(S,t)$ 전체입니다. 가로축은 시간, 세로축은 주가, 색은 ELS 가치입니다.
- 조기상환 평가일(6개월마다)에 상환 기준 위쪽이 **한 색으로 평평하게 덮어써진** 것이 보입니다(그날 상환금으로 고정).
- 왼쪽(아직 낙인 안 됨) 격자에서 50 아래 영역은 오른쪽(낙인됨) 격자와 같은 값입니다. 매일 그 칸을 낙인 격자 값으로 바꿔 쓰기 때문입니다.
- 낙인 격자는 주가가 낮을 때 값이 훨씬 작습니다. 원금 보장이 사라졌기 때문입니다.

**두 번째 그림 (애니메이션):** 한 시점의 세로 줄(가격곡선)을 만기에서 오늘까지 거꾸로 재생합니다. ▶️ Play를 누르세요.
- 평가일에 도착하는 순간 상환 기준 위쪽이 **수평선으로 탁 붙는** 순간을 보세요(덮어쓰기).
- 평가일 사이에서는 BS PDE 때문에 꺾인 곳들이 점점 매끈해집니다.
- 초록(낙인 전)과 빨강(낙인 후) 곡선의 차이가 곧 **"아직 낙인되지 않았다"는 사실의 가치**입니다.

In [ ]:
keep = els_grid["S"] <= 200
Sv, tv = els_grid["S"][keep], els_grid["t"]
fig = make_subplots(rows=1, cols=2, subplot_titles=("V_noKI(S, t): not knocked in yet", "V_KI(S, t): already knocked in"),
                    horizontal_spacing=0.08)
for col, grid_vals in enumerate([els_grid["no"], els_grid["ki"]], start=1):
    fig.add_trace(go.Heatmap(x=tv, y=Sv, z=grid_vals[:, keep].T, colorscale='Viridis', zmin=0, zmax=120,
                             colorbar=dict(title='value', x=1.02), showscale=(col == 2),
                             hovertemplate='t = %{x:.2f}<br>S = %{y:.0f}<br>V = %{z:.2f}<extra></extra>'), row=1, col=col)
    t_prev = 0.0
    for t_k, B_k in zip(els["obs_times"], els["barriers"]):
        fig.add_trace(go.Scatter(x=[t_prev, t_k], y=[B_k, B_k], mode='lines', line=dict(color='white', width=2),
                                 showlegend=False, hoverinfo='skip'), row=1, col=col)
        t_prev = t_k
    fig.add_hline(y=els["ki"], line=dict(color='rgb(255, 75, 75)', dash='dash', width=2), row=1, col=col)
    fig.update_xaxes(title_text='time t (years)', dtick=0.5, row=1, col=col)
    fig.update_yaxes(title_text='underlying S', row=1, col=col)
fig.update_layout(width=1400, height=560, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  title="Both grids after solving backward (white: redemption levels, red dashed: knock-in)")
fig.show()

In [ ]:
# Replay the backward solve: one frame every 5 trading days, plus every observation date
days_total = len(els_grid["t"]) - 1
obs_days = [int(round(t_k * els["days_per_year"])) for t_k in els["obs_times"]]
frame_days = sorted(set(range(days_total, -1, -5)) | set(obs_days) | {0}, reverse=True)

def next_obs(t):
    for t_k, B_k in zip(els["obs_times"], els["barriers"]):
        if t_k >= t - 1e-9:
            return t_k, B_k
    return els["obs_times"][-1], els["barriers"][-1]

def caption(d):
    t = els_grid["t"][d]
    t_k, B_k = next_obs(t)
    on_date = any(abs(t - x) < 1e-9 for x in els["obs_times"])
    note = (f"<b>observation date</b>: S ≥ {B_k} overwritten with {100 * (1 + els['coupon'] * t_k):.0f}" if on_date
            else f"next observation t = {t_k:.1f} (level {B_k})")
    return f"t = {t:.3f} years (time left {els['T'] - t:.3f}):  {note}"

def level_line(d):
    _, B_k = next_obs(els_grid["t"][d])
    return dict(x=[B_k, B_k], y=[-5, 125])

fig = go.Figure([
    go.Scatter(x=Sv, y=els_grid["no"][days_total, keep], mode='lines', line=dict(color='rgb(57, 255, 20)', width=3), name='V_noKI (not knocked in)'),
    go.Scatter(x=Sv, y=els_grid["ki"][days_total, keep], mode='lines', line=dict(color='rgb(255, 75, 75)', width=3), name='V_KI (knocked in)'),
    go.Scatter(mode='lines', line=dict(color='white', dash='dot', width=1.5), name='current redemption level', **level_line(days_total)),
    go.Scatter(x=[els["ki"]] * 2, y=[-5, 125], mode='lines', line=dict(color='rgb(255, 75, 75)', dash='dash', width=1.5), name='knock-in 50'),
])
fig.frames = [go.Frame(name=str(d), traces=[0, 1, 2],
                       data=[go.Scatter(y=els_grid["no"][d, keep]), go.Scatter(y=els_grid["ki"][d, keep]), go.Scatter(**level_line(d))],
                       layout=go.Layout(title_text=caption(d)))
              for d in frame_days]
fig.update_layout(
    title_text=caption(days_total), width=1200, height=600, margin=dict(b=120),
    plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
    xaxis=dict(title='underlying S', range=[0, 200], showgrid=True, gridcolor='rgba(128,128,128,0.2)'),
    yaxis=dict(title='ELS value (per 100)', range=[-5, 125], showgrid=True, gridcolor='rgba(128,128,128,0.2)'),
    legend=dict(x=0.62, y=0.3, bgcolor='rgba(0,0,0,0.4)'),
    updatemenus=[{'type': 'buttons', 'showactive': False, 'x': 0.0, 'y': -0.18, 'xanchor': 'left', 'direction': 'left',
                  'buttons': [{'label': '▶️ Play', 'method': 'animate',
                               'args': [None, {'frame': {'duration': 120, 'redraw': True}, 'fromcurrent': True}]},
                              {'label': '⏸ Pause', 'method': 'animate',
                               'args': [[None], {'frame': {'duration': 0, 'redraw': False}, 'mode': 'immediate'}]}]}],
    sliders=[{'x': 0.15, 'len': 0.85, 'y': -0.1, 'currentvalue': {'prefix': 'trading day: '},
              'steps': [{'label': str(d), 'method': 'animate',
                         'args': [[str(d)], {'frame': {'duration': 0, 'redraw': True}, 'mode': 'immediate'}]}
                        for d in frame_days]}],
)
fig.show()

##### 📈 오늘의 가격곡선, 델타, 그리고 발행사가 쿠폰을 정하는 방법

**격자의 가장 왼쪽 열(오늘, $t=0$)** 에는 "오늘 주가가 $S$ 라면 ELS 가격은 얼마인가"가 모든 $S$ 에 대해 들어 있습니다. 한 번 계산으로 얻은 것입니다.

- **델타** $\partial V/\partial S$ 도 이웃 칸끼리 빼서 바로 얻습니다. 여기서 델타 1은 "액면 100당 기초자산 1단위(최초가격 기준 액면의 100%어치)를 들고 있는 것"과 같은 노출입니다. ELS를 판 발행사는 이 델타만큼 기초자산을 사고팔며 헤지합니다. 상환 기준 근처에서는 받는 금액이 뚝 끊기기 때문에 델타가 크게 튀고, 헤지가 어려워집니다. 특히 낙인선(50) 바로 위에서는 조금만 더 빠져도 원금 보장이 사라지기 때문에 델타가 1을 넘을 만큼 커집니다(아래 출력의 S = 60).
- **공정 쿠폰:** 쿠폰을 바꿔가며 격자를 여러 번 풀어서 가격이 원하는 값이 되는 쿠폰을 찾습니다(`brentq` 근 찾기). 예를 들어 가격이 액면 100이 되는 쿠폰이 "공정 쿠폰"이고, 실제 발행 쿠폰은 발행사의 비용과 마진만큼 이보다 낮게 정해집니다. 한 번 풀이가 빨라야 이런 반복 계산을 할 수 있습니다.

In [ ]:
from scipy.optimize import brentq

S_all, V0_no, V0_ki = els_grid["S"], els_grid["V_no"], els_grid["V_ki"]
delta_no = np.gradient(V0_no, S_all)
show = S_all <= 200
fig = make_subplots(rows=1, cols=2, subplot_titles=("ELS value today V(S, 0)", "Delta of the not-knocked-in ELS ∂V/∂S (per 1 of S)"))
fig.add_trace(go.Scatter(x=S_all[show], y=V0_no[show], mode='lines', line=dict(color='rgb(57, 255, 20)', width=3), name='not knocked in'), row=1, col=1)
fig.add_trace(go.Scatter(x=S_all[show], y=V0_ki[show], mode='lines', line=dict(color='rgb(255, 75, 75)', width=3), name='knocked in'), row=1, col=1)
fig.add_trace(go.Scatter(x=[100], y=[V0_no[i0]], mode='markers+text', marker=dict(size=12, color='white'),
                         text=[f"today: {V0_no[i0]:.2f}"], textposition='top left', showlegend=False), row=1, col=1)
fig.add_trace(go.Scatter(x=S_all[show], y=delta_no[show], mode='lines', line=dict(color='rgb(0, 191, 255)', width=2), name='delta (not knocked in)'), row=1, col=2)
for col in (1, 2):
    fig.add_vline(x=els["barriers"][0], line=dict(color='white', dash='dot'), row=1, col=col)
    fig.add_vline(x=els["ki"], line=dict(color='rgb(255, 75, 75)', dash='dash'), row=1, col=col)
    fig.update_xaxes(title_text='underlying S today', showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
    fig.update_yaxes(showgrid=True, gridcolor='rgba(128,128,128,0.2)', row=1, col=col)
fig.update_layout(width=1400, height=480, plot_bgcolor='rgba(0,0,0,0)', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='white'),
                  legend=dict(x=0.01, y=0.98, bgcolor='rgba(0,0,0,0)'))
fig.show()

for s_now in (120, 100, 80, 60):
    k_ = int(round(s_now / els_grid["dS"]))
    print(f"S today = {s_now:>3}: value {V0_no[k_]:6.2f} (not knocked in) / {V0_ki[k_]:6.2f} (knocked in),  delta {delta_no[k_]:.3f}")

def price_with_coupon(cpn):
    return price_stepdown_els(dict(els, coupon=cpn))["V_no"][i0]
fair = brentq(lambda cpn: price_with_coupon(cpn) - 100.0, 0.0, 0.30, xtol=1e-5)
fair_margin = brentq(lambda cpn: price_with_coupon(cpn) - 99.0, 0.0, 0.30, xtol=1e-5)
print(f"\nCoupon {els['coupon']:.1%} p.a. → price {V0_no[i0]:.3f}")
print(f"Fair coupon (price = 100)                       : {fair:.2%} p.a.")
print(f"Coupon leaving the issuer 1 of value (price = 99): {fair_margin:.2%} p.a.")

**ELS 예제 정리**
1. ELS는 조기상환(특정 날짜의 조건)과 낙인(경로 의존) 때문에 **공식이 없습니다.**
2. 격자에서는 **만기 상환금에서 출발해 뒤로 가며**, 평가일 사이에는 BS PDE 한 스텝을, **평가일에는 상환 조건으로 격자 값을 덮어쓰고**, **낙인은 격자 두 장(낙인 전/후)을 매일 연결**해서 처리합니다.
3. 결과는 몬테카를로와 일치하고, 격자 한 번으로 모든 주가에서의 가격과 델타가 나오며, 공정 쿠폰 같은 반복 계산도 빠릅니다.

**이 예제의 단순화 (실제 상품과의 차이)**
- 기초자산이 1개입니다. 실제 ELS는 대부분 2~3개(예: 지수 3개 중 가장 나쁜 것 기준)라서 2개면 2차원 격자(ADI 등), 3개 이상이면 몬테카를로를 많이 씁니다.
- 변동성이 상수(BS)입니다. 실무에서는 변동성 스마일을 반영하는 local volatility 모델을 격자에 넣습니다. 계수 $a_i, b_i, c_i$ 에 $\sigma$ 대신 $\sigma(S_i, t_j)$ 를 넣으면 됩니다.
- 배당, 발행사 신용위험, 평가일의 영업일 조정 등은 생략했습니다.
- explicit 방법은 안정성 때문에 스텝 수가 많습니다(여기서 21,168개). 실무에서는 큰 시간 간격을 쓸 수 있는 implicit / Crank–Nicolson을 주로 씁니다. 매 스텝 푸는 연립방정식은 삼중대각이라 빠릅니다.

###### ______________________________________________________________________________________________________________________________________

#### 📝 정리

1. **옵션 가격 = PDE의 해.** 델타헤지 논리로 무작위성을 지우면 BS PDE(또는 Heston PDE)가 남습니다. 해석해가 없으면 수치적으로 풉니다.
2. **유한차분의 전부**: 미분을 차분으로 바꾸고($f' \approx \frac{f(x+h)-f(x)}{h}$), 모르는 값에 대해 풀어서, 아는 곳에서 모르는 곳으로 한 칸씩 채워 나갑니다.
3. **BS는 끝(만기 payoff)에서 출발해 시간을 거꾸로** 계산합니다. 경계조건 $V(0,t)=0$, $V(S_{max},t) = S_{max}-Ke^{-r(T-t)}$ 이 꼭 필요합니다. 원본은 위쪽 경계를 빠뜨려 $S_{max}=150$ 결과가 틀렸습니다.
4. **격자의 칸 하나 = 한 스텝 뒤의 이웃 세 칸의 가중합** $a V_\downarrow + b V_\to + c V_\uparrow$ 입니다.
5. 그 가중치는 **위험중립 삼항트리의 할인된 확률**입니다(drift $r$, 분산 $\sigma^2S^2\Delta t$). 그래서 유한차분, 몬테카를로, 해석해가 같은 값을 줍니다(Feynman–Kac).
6. **안정성 = 가중치(확률)가 음수가 아닐 것**: 열방정식은 $\lambda \le \tfrac12$, BS explicit은 $\Delta t \lesssim 1/(\sigma^2M^2)$. 어기면 오차가 폭발합니다.
7. **정확도**: $\Delta S$ 를 반으로 줄이면 오차는 약 1/4, 계산량은 약 8배입니다.
8. **왜 쓰나:** 격자 한 번으로 모든 주가의 가격과 **그릭스**가 나오고, 아메리칸 옵션처럼 **공식이 없는 상품**도 한 줄만 추가하면 풀립니다. 공식이 있는 BS는 방법을 검증하는 시험대였습니다.
9. **실전 (스텝다운 ELS):** 평가일에는 상환 조건으로 격자 값을 **덮어쓰고**, 낙인 같은 경로 의존은 **격자를 상태별로 여러 장** 두어 연결하면, 공식이 없는 구조화상품도 같은 격자 방법으로 평가됩니다.

---

##### 6.) 💭 마무리와 다음 주제

퀀트 금융에서는 모델(Black-Scholes, Heston, …)을 정하고 헤지 논리를 적용해 **가격결정 PDE**를 얻습니다.

미분의 정의 하나만으로 이 PDE의 해, 즉 그 모델에서의 가격함수를 근사할 수 있습니다.

상품이 달라져도 방정식은 그대로이고, **종료조건과 경계조건**이 바뀝니다. 예를 들어 콜에서 풋으로 바꾸면 payoff와 경계만 바뀌고, 아메리칸은 매 스텝 조기행사 조건이 추가되며, 배리어 옵션은 배리어 위치에 경계조건이 걸립니다.

같은 가격함수를 다른 논리와 기법으로도 구할 수 있습니다. 대표적인 것이 **몬테카를로 시뮬레이션**이고, 이 노트북에서 두 방법이 같은 값을 준다는 것도 확인했습니다.

**다음 주제 (원 저자의 예고)**

- 위험중립 가격결정, 측도 변환, 이항트리

- Black-Scholes 방정식 유도 (델타 헤지)

- Heston 방정식 유도 (베가 헤지 후 델타 헤지)

- 더 발전된 PDE 근사 코드 (Black-Scholes, Heston, …)

- 수치기법(scheme), 조건, 오차 분석 (재미는 덜하지만 꼭 필요함)